In [ ]:
# ============================================================
# EXPLORATION ET PRÉTRAITEMENT DU DATASET ALZHEIMER EXISTANT
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm import tqdm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURATION DES CHEMINS
# ============================================================

BASE_DIR = "C:/Users/angej/Desktop/MultimodalAI/Alzheimer/data"
CLEANED_DIR = os.path.join(BASE_DIR, "cleaned")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")

# Chemins spécifiques selon votre structure
TRAIN_DIR = os.path.join(PROCESSED_DIR, "train")
TEST_DIR = os.path.join(PROCESSED_DIR, "test")

CLASSES_CSV = os.path.join(PROCESSED_DIR, "test", "_classes.csv") 

CONFIG = {
    'train_dir': TRAIN_DIR,
    'test_dir': TEST_DIR,
    'classes_csv': CLASSES_CSV,
    'img_size': 224,
    'batch_size': 16,
    'num_workers': 6,
}



In [ ]:
import os

# Créer le dossier figures s'il n'existe pas
FIGURES_DIR = os.path.join('..', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"📁 Dossier figures: {os.path.abspath(FIGURES_DIR)}")
print(f"   Existe: {os.path.exists(FIGURES_DIR)}")


In [ ]:

# ============================================================
# 2. EXPLORATION COMPLÈTE DU DATASET
# ============================================================

def explore_dataset_structure():
    """Explore la structure existante du dataset"""
    print("🔍 Exploration de la structure des données...")
    
    # Vérification de l'existence des dossiers
    if not os.path.exists(TRAIN_DIR):
        print(f"❌ Dossier train introuvable: {TRAIN_DIR}")
        return None, None, None
    
    if not os.path.exists(TEST_DIR):
        print(f"❌ Dossier test introuvable: {TEST_DIR}")
        return None, None, None
    
    # Liste des classes (dossiers dans train)
    try:
        classes = [d for d in os.listdir(TRAIN_DIR) 
                  if os.path.isdir(os.path.join(TRAIN_DIR, d)) and not d.startswith('.')]
        classes.sort()
        print(f"📁 Classes détectées: {classes}")
    except Exception as e:
        print(f"❌ Erreur lors de la lecture des classes: {e}")
        return None, None, None
    
    # Comptage des images par classe
    train_counts = {}
    test_counts = {}
    
    print("\n📊 Distribution des images:")
    print("Classe\t\t\tTrain\tTest\tTotal")
    print("-" * 50)
    
    total_train = 0
    total_test = 0
    
    for class_name in classes:
        train_class_dir = os.path.join(TRAIN_DIR, class_name)
        test_class_dir = os.path.join(TEST_DIR, class_name)
        
        train_count = len(os.listdir(train_class_dir)) if os.path.exists(train_class_dir) else 0
        test_count = len(os.listdir(test_class_dir)) if os.path.exists(test_class_dir) else 0
        
        train_counts[class_name] = train_count
        test_counts[class_name] = test_count
        total_train += train_count
        total_test += test_count
        
        print(f"{class_name:20} {train_count:5} {test_count:5} {train_count + test_count:5}")
    
    print("-" * 50)
    print(f"{'TOTAL':20} {total_train:5} {total_test:5} {total_train + total_test:5}")
    
    return classes, train_counts, test_counts

# Exploration de la structure
classes, train_counts, test_counts = explore_dataset_structure()

if classes is None:
    print("❌ Impossible de charger la structure des données")
    exit()


In [ ]:

# ============================================================
# 3. ANALYSE DES IMAGES (DIMENSIONS, INTENSITÉS)
# ============================================================

def analyze_images_stats(sample_size=200):
    """Analyse statistique des images"""
    print("\n📈 Analyse des dimensions et intensités des images...")
    
    all_widths = []
    all_heights = []
    all_intensities = []
    all_aspect_ratios = []
    
    analyzed_classes = {}
    
    for class_name in classes:
        class_dir = os.path.join(TRAIN_DIR, class_name)
        if not os.path.exists(class_dir):
            continue
            
        images = os.listdir(class_dir)
        if not images:
            continue
            
        # Échantillon pour vitesse
        sample_images = images[:min(sample_size, len(images))]
        class_widths = []
        class_heights = []
        class_intensities = []
        
        for img_name in tqdm(sample_images, desc=f"Analyse {class_name}"):
            img_path = os.path.join(class_dir, img_name)
            try:
                with Image.open(img_path) as img:
                    # Conversion en RGB si nécessaire
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    
                    # Dimensions
                    width, height = img.size
                    class_widths.append(width)
                    class_heights.append(height)
                    all_widths.append(width)
                    all_heights.append(height)
                    all_aspect_ratios.append(width / height)
                    
                    # Intensités
                    img_array = np.array(img)
                    intensity = np.mean(img_array)
                    class_intensities.append(intensity)
                    all_intensities.append(intensity)
                    
            except Exception as e:
                print(f"❌ Erreur avec {img_path}: {e}")
                continue
        
        analyzed_classes[class_name] = {
            'width_mean': np.mean(class_widths) if class_widths else 0,
            'height_mean': np.mean(class_heights) if class_heights else 0,
            'intensity_mean': np.mean(class_intensities) if class_intensities else 0,
            'sample_size': len(sample_images)
        }
    
    # Statistiques globales
    print(f"\n📏 DIMENSIONS MOYENNES:")
    print(f"   Largeur: {np.mean(all_widths):.1f} ± {np.std(all_widths):.1f} px")
    print(f"   Hauteur: {np.mean(all_heights):.1f} ± {np.std(all_heights):.1f} px")
    print(f"   Ratio largeur/hauteur: {np.mean(all_aspect_ratios):.2f} ± {np.std(all_aspect_ratios):.2f}")
    
    print(f"\n🎨 INTENSITÉS MOYENNES:")
    print(f"   Intensité globale: {np.mean(all_intensities):.2f} ± {np.std(all_intensities):.2f}")
    
    print(f"\n📊 PAR CLASSE:")
    for class_name, stats in analyzed_classes.items():
        print(f"   {class_name}: {stats['width_mean']:.0f}x{stats['height_mean']:.0f} px, "
              f"intensité: {stats['intensity_mean']:.1f}")
    
    return all_widths, all_heights, all_intensities, all_aspect_ratios

# Analyse statistique
widths, heights, intensities, aspect_ratios = analyze_images_stats()


In [ ]:
# ============================================================
# 2. LECTURE DU FICHIER DE CLASSES
# ============================================================

def read_classes_csv(csv_path):
    """Lit le fichier _classes.csv pour comprendre la structure"""
    print("📖 Lecture du fichier _classes.csv...")
    
    if not os.path.exists(csv_path):
        print(f"❌ Fichier _classes.csv introuvable: {csv_path}")
        print(f"   Vérifiez le chemin: {csv_path}")
        return None
    
    try:
        # Essayer différents séparateurs
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(csv_path, sep=sep)
                if len(df.columns) >= 2:
                    print(f"✅ Fichier lu avec séparateur: '{sep}'")
                    print(f"   Colonnes: {list(df.columns)}")
                    print(f"   Lignes: {len(df)}")
                    return df
            except:
                continue
        
        # Si aucun séparateur ne fonctionne, essayer sans header
        df = pd.read_csv(csv_path, header=None)
        print(f"✅ Fichier lu sans header")
        print(f"   Colonnes: {len(df.columns)}")
        print(f"   Lignes: {len(df)}")
        return df
        
    except Exception as e:
        print(f"❌ Erreur lecture _classes.csv: {e}")
        return None

# Lecture du fichier classes
classes_df = read_classes_csv(CONFIG['classes_csv'])


In [ ]:

# ============================================================
# 3. EXPLORATION DE LA STRUCTURE RÉELLE
# ============================================================

def explore_actual_structure():
    """Explore la structure réelle des données"""
    print("\n🔍 Exploration de la structure réelle...")
    
    # Vérifier la structure train
    train_structure = {}
    if os.path.exists(CONFIG['train_dir']):
        print(f"📁 Structure TRAIN:")
        train_items = os.listdir(CONFIG['train_dir'])
        for item in train_items:
            item_path = os.path.join(CONFIG['train_dir'], item)
            if os.path.isdir(item_path):
                # Compter les images dans le dossier de classe
                images = [f for f in os.listdir(item_path) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                train_structure[item] = len(images)
                print(f"   {item}: {len(images)} images")
            else:
                print(f"   {item}: fichier")
    else:
        print(f"❌ Dossier train introuvable: {CONFIG['train_dir']}")
    
    # Vérifier la structure test
    test_structure = {}
    test_images_count = 0
    test_images_list = []
    
    if os.path.exists(CONFIG['test_dir']):
        print(f"\n📁 Structure TEST:")
        test_items = os.listdir(CONFIG['test_dir'])
        
        # Compter les images dans test
        test_images = [f for f in test_items 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        test_images_count = len(test_images)
        test_images_list = test_images
        print(f"   Images dans test: {test_images_count}")
        
        # Afficher les sous-dossiers
        subdirs = [f for f in test_items if os.path.isdir(os.path.join(CONFIG['test_dir'], f))]
        if subdirs:
            print(f"   Sous-dossiers: {subdirs}")
        
        # Afficher d'autres fichiers
        other_files = [f for f in test_items 
                    if not f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')) and 
                    not os.path.isdir(os.path.join(CONFIG['test_dir'], f))]
        for file in other_files:
            print(f"   {file}: fichier")
    else:
        print(f"❌ Dossier test introuvable: {CONFIG['test_dir']}")
    
    return train_structure, test_images_count, test_images_list


# Exploration
train_structure, test_images_count, test_images_list = explore_actual_structure()


In [ ]:

# ============================================================
# 4. ANALYSE AVEC LE FICHIER CLASSES.CSV
# ============================================================

def analyze_with_classes_csv(classes_df, test_images_list):
    """Analyse la structure en utilisant le fichier classes.csv"""
    print("\n📊 Analyse avec le fichier de classes...")
    
    if classes_df is not None:
        print("📋 Contenu du fichier _classes.csv:")
        print(classes_df.head(10))
        print(f"\nShape: {classes_df.shape}")
        
        # Essayer de deviner les colonnes
        if len(classes_df.columns) >= 2:
            # Prendre les deux premières colonnes comme fichier et classe
            file_col = classes_df.columns[0]
            class_col = classes_df.columns[1]
            
            print(f"📁 Colonne fichiers: {file_col}")
            print(f"🏷️  Colonne classes: {class_col}")
            
            # Statistiques des classes
            class_distribution = classes_df[class_col].value_counts()
            print(f"\n📊 Distribution des classes dans le CSV:")
            for class_name, count in class_distribution.items():
                print(f"   {class_name}: {count} images")
            
            # Vérifier la correspondance avec les images réelles
            if test_images_list:
                csv_files = set(classes_df[file_col].astype(str))
                actual_files = set(test_images_list)
                
                matching_files = csv_files.intersection(actual_files)
                missing_in_csv = actual_files - csv_files
                missing_in_folder = csv_files - actual_files
                
                print(f"\n🔍 Correspondance fichiers:")
                print(f"   Fichiers correspondants: {len(matching_files)}")
                print(f"   Fichiers manquants dans CSV: {len(missing_in_csv)}")
                print(f"   Fichiers manquants dans dossier: {len(missing_in_folder)}")
                
                if missing_in_csv:
                    print(f"   ⚠️  Premiers fichiers manquants dans CSV: {list(missing_in_csv)[:5]}")
                if missing_in_folder:
                    print(f"   ⚠️  Premiers fichiers manquants dans dossier: {list(missing_in_folder)[:5]}")
            
            return class_distribution, file_col, class_col
        else:
            print("❌ Le fichier CSV n'a pas assez de colonnes")
    
    return None, None, None

class_distribution, file_col, class_col = analyze_with_classes_csv(classes_df, test_images_list)


In [ ]:

# ============================================================
# 5. VÉRIFICATION DE LA STRUCTURE PROCESSED
# ============================================================

def verify_processed_structure():
    """Vérifie si la structure processed est déjà organisée correctement"""
    print("\n🔍 Vérification de la structure processed...")
    
    # Vérifier la structure train
    train_classes = []
    train_stats = {}
    if os.path.exists(CONFIG['train_dir']):
        train_items = os.listdir(CONFIG['train_dir'])
        for item in train_items:
            item_path = os.path.join(CONFIG['train_dir'], item)
            if os.path.isdir(item_path):
                images = [f for f in os.listdir(item_path) 
                         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                if images:
                    train_classes.append(item)
                    train_stats[item] = len(images)
                    print(f"✅ Train - {item}: {len(images)} images")
    
    # Vérifier la structure test
    test_classes = []
    test_stats = {}
    if os.path.exists(CONFIG['test_dir']):
        test_items = os.listdir(CONFIG['test_dir'])
        for item in test_items:
            item_path = os.path.join(CONFIG['test_dir'], item)
            if os.path.isdir(item_path):
                images = [f for f in os.listdir(item_path) 
                         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                if images:
                    test_classes.append(item)
                    test_stats[item] = len(images)
                    print(f"✅ Test - {item}: {len(images)} images")
            else:
                # Si c'est un fichier image directement dans test/
                if item.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    print(f"📄 Test - Image directe: {item}")
    
    return train_classes, test_classes, train_stats, test_stats

train_classes, test_classes, train_stats, test_stats = verify_processed_structure()


In [ ]:

def organize_test_set_if_needed():
    """Organise le test set si ce n'est pas déjà fait"""
    
    # Si le test set n'a pas de sous-dossiers de classe mais qu'on a le CSV et des images directes
    if not test_classes and test_images_count > 0 and classes_df is not None and file_col and class_col:
        print("\n🔄 Organisation du test set...")
        
        # Créer les dossiers de classe dans test
        unique_classes = classes_df[class_col].unique()
        for class_name in unique_classes:
            class_dir = os.path.join(CONFIG['test_dir'], str(class_name))
            os.makedirs(class_dir, exist_ok=True)
            print(f"📁 Créé: {class_dir}")
        
        # Déplacer les images dans les bons dossiers
        moved_count = 0
        for idx, row in classes_df.iterrows():
            image_filename = str(row[file_col])
            source_path = os.path.join(CONFIG['test_dir'], image_filename)
            target_dir = os.path.join(CONFIG['test_dir'], str(row[class_col]))
            
            # Essayer différentes extensions
            moved = False
            for ext in ['', '.jpg', '.jpeg', '.png', '.bmp']:
                actual_source = source_path + ext
                if os.path.exists(actual_source):
                    try:
                        import shutil
                        target_path = os.path.join(target_dir, os.path.basename(actual_source))
                        shutil.move(actual_source, target_path)
                        moved_count += 1
                        moved = True
                        break
                    except Exception as e:
                        print(f"❌ Erreur déplacement {actual_source}: {e}")
            
            if not moved:
                print(f"⚠️  Image non trouvée: {image_filename}")
        
        print(f"✅ {moved_count} images organisées dans le test set")
        return True
    else:
        print("✅ Structure test déjà organisée ou impossible à organiser")
        return False

# Organiser si nécessaire
organized = organize_test_set_if_needed()

# Si organisation effectuée, re-vérifier la structure
if organized:
    train_classes, test_classes, train_stats, test_stats = verify_processed_structure()

# ============================================================
# 7. ANALYSE DES DONNÉES FINALES
# ============================================================

def analyze_final_dataset():
    """Analyse finale du dataset organisé"""
    print("\n📊 ANALYSE FINALE DU DATASET")
    print("=" * 50)
    
    # Statistiques train
    total_train = sum(train_stats.values()) if train_stats else 0
    if train_stats:
        print("📁 TRAIN SET:")
        for class_name, count in train_stats.items():
            print(f"   {class_name}: {count} images")
        print(f"   TOTAL TRAIN: {total_train} images")
    else:
        print("❌ Aucune donnée dans le train set")
    
    # Statistiques test
    total_test = sum(test_stats.values()) if test_stats else 0
    if test_stats:
        print("\n📁 TEST SET:")
        for class_name, count in test_stats.items():
            print(f"   {class_name}: {count} images")
        print(f"   TOTAL TEST: {total_test} images")
    else:
        print("❌ Aucune donnée dans le test set")
    
    # Analyse du déséquilibre
    if train_stats:
        train_values = list(train_stats.values())
        if min(train_values) > 0:
            train_imbalance = max(train_values) / min(train_values)
            print(f"\n⚖️  Déséquilibre Train: {train_imbalance:.2f}:1")
        else:
            print(f"⚠️  Certaines classes train ont 0 image")
    
    if test_stats:
        test_values = list(test_stats.values())
        if min(test_values) > 0:
            test_imbalance = max(test_values) / min(test_values)
            print(f"⚖️  Déséquilibre Test: {test_imbalance:.2f}:1")
        else:
            print(f"⚠️  Certaines classes test ont 0 image")
    
    return train_stats, test_stats

train_stats, test_stats = analyze_final_dataset()


In [ ]:

# ============================================================
# 8. ANALYSE DES CARACTÉRISTIQUES DES IMAGES
# ============================================================

def analyze_image_characteristics(sample_size=50):
    """Analyse les caractéristiques des images"""
    print("\n📈 ANALYSE DES CARACTÉRISTIQUES DES IMAGES")
    print("=" * 50)
    
    all_widths = []
    all_heights = []
    all_intensities = []
    
    # Analyser quelques images de chaque classe du train set
    if train_stats:
        for class_name in train_stats.keys():
            class_dir = os.path.join(CONFIG['train_dir'], class_name)
            if os.path.isdir(class_dir):
                images = [f for f in os.listdir(class_dir) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                
                if not images:
                    continue
                
                sample_images = images[:min(sample_size, len(images))]
                print(f"🔍 Analyse {class_name} ({len(sample_images)} images)...")
                
                for img_name in tqdm(sample_images, desc=f"  {class_name}"):
                    img_path = os.path.join(class_dir, img_name)
                    try:
                        with Image.open(img_path) as img:
                            # Conversion RGB si nécessaire
                            if img.mode != 'RGB':
                                img = img.convert('RGB')
                            
                            # Dimensions
                            width, height = img.size
                            all_widths.append(width)
                            all_heights.append(height)
                            
                            # Intensité
                            img_array = np.array(img)
                            intensity = np.mean(img_array)
                            all_intensities.append(intensity)
                            
                    except Exception as e:
                        print(f"❌ Erreur avec {img_path}: {e}")
                        continue
    
    # Statistiques
    if all_widths:
        print(f"\n📏 DIMENSIONS MOYENNES:")
        print(f"   Largeur: {np.mean(all_widths):.1f} ± {np.std(all_widths):.1f} px")
        print(f"   Hauteur: {np.mean(all_heights):.1f} ± {np.std(all_heights):.1f} px")
        print(f"   Ratio: {np.mean(all_widths)/np.mean(all_heights):.2f}")
        
        print(f"\n🎨 INTENSITÉS:")
        print(f"   Moyenne: {np.mean(all_intensities):.2f} ± {np.std(all_intensities):.2f}")
        print(f"   Min: {np.min(all_intensities):.2f}, Max: {np.max(all_intensities):.2f}")
        
        return all_widths, all_heights, all_intensities
    else:
        print("❌ Aucune image trouvée pour l'analyse")
        return [], [], []

widths, heights, intensities = analyze_image_characteristics()


In [ ]:

# ============================================================
# 9. RAPPORT FINAL
# ============================================================

print("\n" + "="*70)
print("📋 RAPPORT FINAL - DATASET PROCESSED")
print("="*70)

print(f"\n📍 STRUCTURE UTILISÉE: {PROCESSED_DIR}")

print(f"\n📊 RÉSUMÉ DES DONNÉES:")
if train_stats:
    print(f"   TRAIN: {sum(train_stats.values())} images, {len(train_stats)} classes")
    for class_name, count in train_stats.items():
        print(f"     - {class_name}: {count} images")

if test_stats:
    print(f"   TEST: {sum(test_stats.values())} images, {len(test_stats)} classes")
    for class_name, count in test_stats.items():
        print(f"     - {class_name}: {count} images")

if widths:
    print(f"\n📏 CARACTÉRISTIQUES DES IMAGES:")
    print(f"   Dimensions: {np.mean(widths):.0f} x {np.mean(heights):.0f} px")
    print(f"   Intensité moyenne: {np.mean(intensities):.1f}")

print(f"\n✅ ÉTAT DU DATASET:")
if train_stats and test_stats:
    print("   🟢 PRÊT POUR L'ENTRAÎNEMENT")
    print(f"\n🎯 RECOMMANDATIONS:")
    print(f"   1. Taille d'image: {CONFIG['img_size']}x{CONFIG['img_size']} px")
    print(f"   2. Normalisation ImageNet")
    print(f"   3. Gestion du déséquilibre avec WeightedRandomSampler")
else:
    print("   🔴 PROBLÈMES DÉTECTÉS - Vérifier la structure des dossiers")

print(f"\n📁 Dossiers à utiliser pour l'entraînement:")
print(f"   Train: {CONFIG['train_dir']}")
print(f"   Test:  {CONFIG['test_dir']}")

print("\n" + "="*70)
print("✨ ANALYSE TERMINÉE!")
print("="*70)

In [ ]:
# ============================================================
# 8. VISUALISATION COMPLÈTE
# ============================================================

def calculate_imbalance(stats):
    """Calcule le ratio de déséquilibre"""
    if not stats:
        return 0
    values = [v for v in stats.values() if v > 0]
    if not values or min(values) == 0:
        return 0
    return max(values) / min(values)

def visualize_exploration(train_stats, test_stats, widths, heights, intensities):
    """Visualisations complètes de l'exploration"""
    
    # Calculer le déséquilibre
    train_imbalance = calculate_imbalance(train_stats)
    test_imbalance = calculate_imbalance(test_stats)
    
    # Préparer les données pour les graphiques
    classes = list(train_stats.keys()) if train_stats else list(test_stats.keys()) if test_stats else []
    train_counts = [train_stats.get(cls, 0) for cls in classes] if train_stats else []
    test_counts = [test_stats.get(cls, 0) for cls in classes] if test_stats else []
    
    # Calculer les ratios d'aspect
    aspect_ratios = [w/h for w, h in zip(widths, heights)] if widths and heights else []
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('📊 EXPLORATION COMPLÈTE - DATASET ALZHEIMER', fontsize=16, fontweight='bold')
    
    # 1. Distribution des classes (train)
    if train_counts:
        bars1 = axes[0, 0].bar(classes, train_counts, color='skyblue', alpha=0.7, label='Train')
        axes[0, 0].set_title('Distribution des classes (Train)', fontweight='bold')
        axes[0, 0].tick_params(axis='x', rotation=45)
        axes[0, 0].set_ylabel('Nombre d\'images')
        # Ajouter les valeurs sur les barres
        for bar in bars1:
            height = bar.get_height()
            if height > 0:
                axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                               f'{int(height)}', ha='center', va='bottom')
    else:
        axes[0, 0].text(0.5, 0.5, 'Aucune donnée Train', 
                       ha='center', va='center', transform=axes[0, 0].transAxes, fontsize=12)
        axes[0, 0].set_title('Distribution des classes (Train)', fontweight='bold')
    
    # 2. Distribution des classes (test)
    if test_counts:
        bars2 = axes[0, 1].bar(classes, test_counts, color='lightcoral', alpha=0.7, label='Test')
        axes[0, 1].set_title('Distribution des classes (Test)', fontweight='bold')
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].set_ylabel('Nombre d\'images')
        for bar in bars2:
            height = bar.get_height()
            if height > 0:
                axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                               f'{int(height)}', ha='center', va='bottom')
    else:
        axes[0, 1].text(0.5, 0.5, 'Aucune donnée Test', 
                       ha='center', va='center', transform=axes[0, 1].transAxes, fontsize=12)
        axes[0, 1].set_title('Distribution des classes (Test)', fontweight='bold')
    
    # 3. Dimensions des images
    if widths and heights:
        axes[0, 2].scatter(widths, heights, alpha=0.6, s=20, color='green')
        axes[0, 2].set_xlabel('Largeur (px)')
        axes[0, 2].set_ylabel('Hauteur (px)')
        axes[0, 2].set_title('Distribution des dimensions', fontweight='bold')
        axes[0, 2].grid(True, alpha=0.3)
        # Ajouter les moyennes
        axes[0, 2].axvline(np.mean(widths), color='red', linestyle='--', alpha=0.7, 
                          label=f'Largeur moy: {np.mean(widths):.0f}px')
        axes[0, 2].axhline(np.mean(heights), color='blue', linestyle='--', alpha=0.7, 
                          label=f'Hauteur moy: {np.mean(heights):.0f}px')
        axes[0, 2].legend()
    else:
        axes[0, 2].text(0.5, 0.5, 'Aucune donnée dimensions', 
                       ha='center', va='center', transform=axes[0, 2].transAxes, fontsize=12)
        axes[0, 2].set_title('Distribution des dimensions', fontweight='bold')
    
    # 4. Histogramme des intensités
    if intensities:
        axes[1, 0].hist(intensities, bins=50, color='orange', alpha=0.7, edgecolor='black')
        axes[1, 0].set_xlabel('Intensité moyenne')
        axes[1, 0].set_ylabel('Fréquence')
        axes[1, 0].set_title('Distribution des intensités', fontweight='bold')
        axes[1, 0].axvline(np.mean(intensities), color='red', linestyle='--', 
                          label=f'Moyenne: {np.mean(intensities):.1f}')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].text(0.5, 0.5, 'Aucune donnée intensités', 
                       ha='center', va='center', transform=axes[1, 0].transAxes, fontsize=12)
        axes[1, 0].set_title('Distribution des intensités', fontweight='bold')
    
    # 5. Ratio d'aspect
    if aspect_ratios:
        axes[1, 1].hist(aspect_ratios, bins=30, color='purple', alpha=0.7, edgecolor='black')
        axes[1, 1].set_xlabel('Ratio Largeur/Hauteur')
        axes[1, 1].set_ylabel('Fréquence')
        axes[1, 1].set_title('Distribution des ratios d\'aspect', fontweight='bold')
        axes[1, 1].axvline(np.mean(aspect_ratios), color='red', linestyle='--',
                          label=f'Moyenne: {np.mean(aspect_ratios):.2f}')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    else:
        axes[1, 1].text(0.5, 0.5, 'Aucune donnée ratios', 
                       ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=12)
        axes[1, 1].set_title('Distribution des ratios d\'aspect', fontweight='bold')
    
    # 6. Déséquilibre des classes
    imbalance_text = f"""ANALYSE DU DÉSÉQUILIBRE

Ratio Train: {train_imbalance:.1f}:1
Ratio Test:  {test_imbalance:.1f}:1

Recommandations:
• WeightedRandomSampler
• Class Weights dans la loss
• Focal Loss
• Data Augmentation ciblée
"""
    axes[1, 2].text(0.1, 0.7, imbalance_text, fontsize=11, va='top', 
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.7))
    axes[1, 2].set_title('Analyse du déséquilibre', fontweight='bold')
    axes[1, 2].set_xticks([])
    axes[1, 2].set_yticks([])
    
    plt.tight_layout()
    plt.savefig('../figures/alzheimer_dataset_exploration.png', dpi=300, bbox_inches='tight')
    plt.show()
   

In [ ]:
# ============================================================
# 9. VISUALISATION DES ÉCHANTILLONS D'IMAGES
# ============================================================

def visualize_image_samples(num_samples=8):
    """Visualise des échantillons d'images de chaque classe"""
    
    print("\n🖼️  Visualisation des échantillons d'images...")
    
    if not train_stats:
        print("❌ Aucune donnée train pour la visualisation")
        return
    
    fig, axes = plt.subplots(len(train_stats), num_samples, figsize=(20, 3*len(train_stats)))
    if len(train_stats) == 1:
        axes = [axes]
    
    for i, (class_name, count) in enumerate(train_stats.items()):
        class_dir = os.path.join(CONFIG['train_dir'], class_name)
        if not os.path.exists(class_dir):
            continue
            
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        if not images:
            continue
            
        # Prendre un échantillon d'images
        sample_images = images[:min(num_samples, len(images))]
        
        for j, img_name in enumerate(sample_images):
            img_path = os.path.join(class_dir, img_name)
            try:
                with Image.open(img_path) as img:
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    
                    if len(train_stats) > 1:
                        ax = axes[i, j]
                    else:
                        ax = axes[j]
                    
                    ax.imshow(np.array(img))
                    ax.set_title(f'{class_name}\n{img_name[:15]}...', fontsize=8)
                    ax.axis('off')
                    
            except Exception as e:
                print(f"❌ Erreur avec {img_path}: {e}")
                continue
    
    plt.suptitle('ÉCHANTILLONS D\'IMAGES PAR CLASSE - DATASET ALZHEIMER', 
                 fontsize=16, fontweight='bold', y=0.95)
    plt.tight_layout()
    plt.savefig('../figures/alzheimer_image_samples.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:

# ============================================================
# 10. VISUALISATION DES COURBES DE DISTRIBUTION
# ============================================================

def visualize_distribution_analysis(train_stats, test_stats):
    """Visualisation avancée de la distribution des données"""
    
    if not train_stats and not test_stats:
        print("❌ Aucune donnée pour la visualisation des distributions")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('ANALYSE DE DISTRIBUTION - DATASET ALZHEIMER', fontsize=16, fontweight='bold')
    
    # Distribution train vs test
    if train_stats and test_stats:
        classes = list(train_stats.keys())
        train_values = [train_stats[cls] for cls in classes]
        test_values = [test_stats.get(cls, 0) for cls in classes]
        
        x = np.arange(len(classes))
        width = 0.35
        
        axes[0].bar(x - width/2, train_values, width, label='Train', color='skyblue', alpha=0.7)
        axes[0].bar(x + width/2, test_values, width, label='Test', color='lightcoral', alpha=0.7)
        
        axes[0].set_xlabel('Classes')
        axes[0].set_ylabel('Nombre d\'images')
        axes[0].set_title('Distribution Train vs Test', fontweight='bold')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(classes, rotation=45)
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Ajouter les valeurs
        for i, v in enumerate(train_values):
            axes[0].text(i - width/2, v + max(train_values + test_values)*0.01, str(v), 
                        ha='center', va='bottom', fontsize=8)
        for i, v in enumerate(test_values):
            axes[0].text(i + width/2, v + max(train_values + test_values)*0.01, str(v), 
                        ha='center', va='bottom', fontsize=8)
    
    # Diagramme en camembert de la répartition
    if train_stats:
        sizes = list(train_stats.values())
        labels = [f'{cls}\n({count})' for cls, count in train_stats.items()]
        
        wedges, texts, autotexts = axes[1].pie(sizes, labels=labels, autopct='%1.1f%%',
                                              startangle=90, colors=plt.cm.Set3(np.linspace(0, 1, len(sizes))))
        
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
        
        axes[1].set_title('Répartition des classes (Train)', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../figures/alzheimer_distribution_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:

# ============================================================
# 11. EXÉCUTION DES VISUALISATIONS
# ============================================================

print("\n" + "="*70)
print("🎨 GÉNÉRATION DES VISUALISATIONS")
print("="*70)

# 1. Visualisation principale
print("📊 Génération de la visualisation principale...")
visualize_exploration(train_stats, test_stats, widths, heights, intensities)


In [ ]:

# 2. Visualisation des échantillons d'images
print("🖼️  Génération des échantillons d'images...")
visualize_image_samples()

# 3. Visualisation de la distribution
print("📈 Génération de l'analyse de distribution...")
visualize_distribution_analysis(train_stats, test_stats)

print("\n✅ TOUTES LES VISUALISATIONS ONT ÉTÉ GÉNÉRÉES AVEC SUCCÈS!")
print("📁 Fichiers créés:")
print("   - alzheimer_dataset_exploration.png")
print("   - alzheimer_image_samples.png") 
print("   - alzheimer_distribution_analysis.png")

print("\n" + "="*70)

In [ ]:
# ============================================================
# 6. CRÉATION DES TRANSFORMATIONS
# ============================================================

# Transformations de base pour l'analyse
basic_transform = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
])

# Transformations avec augmentation pour l'entraînement
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet
])

# Transformations de test
test_transform = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\n🛠️ TRANSFORMATIONS CRÉÉES:")
print("   • Redimensionnement: 224x224 px")
print("   • Normalisation: statistiques ImageNet")
print("   • Augmentation: Flip, Rotation, ColorJitter, Affine")


In [ ]:

# ============================================================
# 7. CALCUL DES VARIABLES POUR LE RAPPORT
# ============================================================

def calculate_report_variables(train_stats, test_stats, widths, heights, intensities):
    """Calcule toutes les variables nécessaires pour le rapport"""
    
    # Classes
    classes = list(train_stats.keys()) if train_stats else list(test_stats.keys()) if test_stats else []
    
    # Counts
    train_counts = train_stats if train_stats else {}
    test_counts = test_stats if test_stats else {}
    
    # Déséquilibre
    def get_imbalance(stats):
        if not stats:
            return 0
        values = [v for v in stats.values() if v > 0]
        if not values or min(values) == 0:
            return 0
        return max(values) / min(values)
    
    train_imbalance = get_imbalance(train_stats)
    test_imbalance = get_imbalance(test_stats)
    
    # Ratio d'aspect
    aspect_ratios = [w/h for w, h in zip(widths, heights)] if widths and heights else []
    
    # Classes majoritaires/minoritaires
    majority_class_train = max(train_stats, key=train_stats.get) if train_stats else "N/A"
    minority_class_train = min([cls for cls in train_stats if train_stats[cls] > 0], key=train_stats.get) if train_stats else "N/A"
    
    return {
        'classes': classes,
        'train_counts': train_counts,
        'test_counts': test_counts,
        'train_imbalance': train_imbalance,
        'test_imbalance': test_imbalance,
        'aspect_ratios': aspect_ratios,
        'majority_class_train': majority_class_train,
        'minority_class_train': minority_class_train
    }

# Calcul des variables
report_vars = calculate_report_variables(train_stats, test_stats, widths, heights, intensities)


In [ ]:

# ============================================================
# 8. RAPPORT FINAL D'EXPLORATION
# ============================================================

print("\n" + "="*70)
print("📋 RAPPORT FINAL D'EXPLORATION - DATASET ALZHEIMER")
print("="*70)

print(f"\n🏷️  CLASSES ({len(report_vars['classes'])}):")
for i, class_name in enumerate(report_vars['classes'], 1):
    train_count = report_vars['train_counts'].get(class_name, 0)
    test_count = report_vars['test_counts'].get(class_name, 0)
    print(f"   {i}. {class_name} (Train: {train_count}, Test: {test_count})")

print(f"\n📊 STATISTIQUES GLOBALES:")
total_train = sum(report_vars['train_counts'].values()) if report_vars['train_counts'] else 0
total_test = sum(report_vars['test_counts'].values()) if report_vars['test_counts'] else 0
print(f"   Images d'entraînement: {total_train}")
print(f"   Images de test: {total_test}")
print(f"   Total: {total_train + total_test}")

if widths and heights and intensities:
    print(f"\n📏 CARACTÉRISTIQUES DES IMAGES:")
    print(f"   Dimensions moyennes: {np.mean(widths):.0f} x {np.mean(heights):.0f} px")
    print(f"   Intensité moyenne: {np.mean(intensities):.1f}")
    if report_vars['aspect_ratios']:
        print(f"   Ratio d'aspect moyen: {np.mean(report_vars['aspect_ratios']):.2f}")

print(f"\n⚖️  DÉSÉQUILIBRE:")
max_imbalance = max(report_vars['train_imbalance'], report_vars['test_imbalance'])
print(f"   Ratio maximal: {max_imbalance:.1f}:1")

if report_vars['majority_class_train'] != "N/A":
    majority_count = report_vars['train_counts'][report_vars['majority_class_train']]
    minority_count = report_vars['train_counts'][report_vars['minority_class_train']]
    print(f"   Classe la plus représentée: {report_vars['majority_class_train']} ({majority_count} images)")
    print(f"   Classe la moins représentée: {report_vars['minority_class_train']} ({minority_count} images)")

print(f"\n🎯 RECOMMANDATIONS:")
if max_imbalance > 5:
    print(f"   ⚠️  FORT DÉSÉQUILIBRE DÉTECTÉ - Actions prioritaires:")
    print(f"   1. WeightedRandomSampler OBLIGATOIRE")
    print(f"   2. Class Weights dans la fonction de perte")
    print(f"   3. Data Augmentation intensive pour les classes minoritaires")
    print(f"   4. Focal Loss pour gérer le déséquilibre")
else:
    print(f"   1. Utiliser WeightedRandomSampler pour équilibrer l'entraînement")
    print(f"   2. Appliquer Class Weights dans la fonction de perte")
    print(f"   3. Augmentation de données ciblée pour les classes minoritaires")

print(f"   4. Monitorer la Balanced Accuracy plutôt que l'Accuracy standard")
print(f"   5. Taille d'image recommandée: {CONFIG['img_size']}x{CONFIG['img_size']} px")

print(f"\n✅ PRÉTRAITEMENT RECOMMANDÉ:")
print(f"   • Normalisation ImageNet (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])")
print(f"   • Augmentation: rotation ±15°, flip horizontal (p=0.5)")
print(f"   • ColorJitter: luminosité ±0.2, contraste ±0.2")
print(f"   • RandomAffine: translation ±10%")

print(f"\n📁 STRUCTURE FINALE:")
print(f"   Train: {CONFIG['train_dir']}")
print(f"   Test:  {CONFIG['test_dir']}")

print(f"\n🔧 CONFIGURATION MODÈLE:")
print(f"   Taille d'image: {CONFIG['img_size']}x{CONFIG['img_size']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Device: {'GPU disponible' if torch.cuda.is_available() else 'CPU'}")

print("\n" + "="*70)
print("✨ EXPLORATION TERMINÉE - PRÊT POUR L'ENTRAÎNEMENT!")
print("="*70)

# ============================================================
# 9. SAUVEGARDE DE LA CONFIGURATION
# ============================================================

def save_configuration():
    """Sauvegarde la configuration pour l'entraînement"""
    config_file = os.path.join(PROCESSED_DIR, "dataset_config.json")
    
    config_data = {
        'dataset_info': {
            'total_train_images': total_train,
            'total_test_images': total_test,
            'classes': report_vars['classes'],
            'class_distribution_train': report_vars['train_counts'],
            'class_distribution_test': report_vars['test_counts'],
            'imbalance_ratio': max_imbalance,
            'image_size': CONFIG['img_size']
        },
        'paths': {
            'train_dir': CONFIG['train_dir'],
            'test_dir': CONFIG['test_dir']
        },
        'transforms': {
            'train_augmentations': [
                'Resize', 'RandomHorizontalFlip', 'RandomRotation', 
                'ColorJitter', 'RandomAffine', 'Normalize'
            ],
            'test_augmentations': [
                'Resize', 'Normalize'
            ]
        },
        'recommendations': {
            'use_weighted_sampler': max_imbalance > 3,
            'use_class_weights': True,
            'monitor_balanced_accuracy': True,
            'focal_loss_recommended': max_imbalance > 10
        }
    }
    
    try:
        import json
        with open(config_file, 'w', encoding='utf-8') as f:
            json.dump(config_data, f, indent=2, ensure_ascii=False)
        print(f"\n💾 Configuration sauvegardée: {config_file}")
    except Exception as e:
        print(f"❌ Erreur sauvegarde configuration: {e}")

save_configuration()

# ============================================================
# 10. VÉRIFICATION FINALE
# ============================================================

print(f"\n🔍 VÉRIFICATION FINALE:")
print(f"   ✅ Structure des dossiers: {'OK' if train_stats and test_stats else 'PROBLÈME'}")
print(f"   ✅ Images chargées: {'OK' if total_train + total_test > 0 else 'PROBLÈME'}")
print(f"   ✅ Transformations: OK")
print(f"   ✅ Configuration: OK")

if train_stats and test_stats:
    print(f"\n🎉 VOTRE DATASET ALZHEIMER EST MAINTENANT PRÊT POUR L'ENTRAÎNEMENT!")
    print(f"\nProchaines étapes:")
    print(f"   1. Créer les DataLoaders avec WeightedRandomSampler")
    print(f"   2. Définir le modèle (Vision Transformer recommandé)")
    print(f"   3. Configurer la loss function avec class weights")
    print(f"   4. Lancer l'entraînement en monitorant la balanced accuracy")
else:
    print(f"\n❌ PROBLEMES DÉTECTÉS - Vérifiez la structure de vos dossiers")

print("\n" + "="*70)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURATION
# ============================================================

print("\n" + "="*70)
print("🔍 EXPLORATION DU DATASET ALZHEIMER")
print("="*70)

BASE_DIR = "C:/Users/angej/Desktop/MultimodalAI/Alzheimer/data"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
TRAIN_DIR = os.path.join(PROCESSED_DIR, "train")
TEST_DIR = os.path.join(PROCESSED_DIR, "test")

CONFIG = {
    'train_dir': TRAIN_DIR,
    'test_dir': TEST_DIR,
    'img_size': 224,
    'batch_size': 32,
    'num_workers': 4,
}

print(f"\n📁 Chemins configurés:")
print(f"   Train: {TRAIN_DIR}")
print(f"   Test: {TEST_DIR}")
print(f"   Train existe: {os.path.exists(TRAIN_DIR)}")
print(f"   Test existe: {os.path.exists(TEST_DIR)}")

# ============================================================
# 2. CLASSE DATASET
# ============================================================

class AlzheimerDataset(Dataset):
    """Dataset pour les images Alzheimer"""
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        
        # Vérifier que le dossier existe
        if not os.path.exists(data_dir):
            print(f"⚠️ Dossier introuvable: {data_dir}")
            return
        
        # Charger les classes
        classes = [d for d in os.listdir(data_dir) 
                if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
        classes.sort()
        
        if not classes:
            print(f"⚠️ Aucune classe trouvée dans: {data_dir}")
            return
        
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        self.idx_to_class = {idx: cls for idx, cls in enumerate(classes)}
        
        # Charger les images
        for class_name in classes:
            class_dir = os.path.join(data_dir, class_name)
            images = [f for f in os.listdir(class_dir) 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            
            for img_name in images:
                self.images.append(os.path.join(class_dir, img_name))
                self.labels.append(self.class_to_idx[class_name])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
            
            if self.transform:
                image = self.transform(image)
            
            return image, label
        except Exception as e:
            print(f"Erreur chargement {img_path}: {e}")
            # Retourner une image noire en cas d'erreur
            if self.transform:
                dummy = self.transform(Image.new('RGB', (224, 224)))
            else:
                dummy = Image.new('RGB', (224, 224))
            return dummy, label

# ============================================================
# 3. ANALYSE DE LA STRUCTURE
# ============================================================

print("\n" + "="*70)
print("📂 ANALYSE DE LA STRUCTURE DES DOSSIERS")
print("="*70)

def analyze_directory_structure(data_dir, name):
    """Analyse la structure d'un dossier"""
    print(f"\n{name}:")
    
    if not os.path.exists(data_dir):
        print(f"   ❌ Dossier introuvable")
        return {}
    
    classes = [d for d in os.listdir(data_dir) 
              if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
    
    if not classes:
        print(f"   ⚠️ Aucune classe trouvée")
        return {}
    
    class_distribution = {}
    total_images = 0
    
    for class_name in sorted(classes):
        class_dir = os.path.join(data_dir, class_name)
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        count = len(images)
        class_distribution[class_name] = count
        total_images += count
        print(f"   📁 {class_name}: {count} images")
    
    print(f"   📊 Total: {total_images} images")
    
    return class_distribution

# Analyser train et test
train_distribution = analyze_directory_structure(TRAIN_DIR, "TRAIN SET")
test_distribution = analyze_directory_structure(TEST_DIR, "TEST SET")

# ============================================================
# 4. ANALYSE DU DÉSÉQUILIBRE
# ============================================================

print("\n" + "="*70)
print("⚖️ ANALYSE DU DÉSÉQUILIBRE")
print("="*70)

def analyze_imbalance(distribution, name):
    """Analyse le déséquilibre d'une distribution"""
    
    if not distribution:
        print(f"\n{name}: Pas de données")
        return
    
    print(f"\n{name}:")
    
    counts = list(distribution.values())
    class_names = list(distribution.keys())
    
    max_count = max(counts)
    min_count = min(counts)
    total = sum(counts)
    
    max_class = class_names[counts.index(max_count)]
    min_class = class_names[counts.index(min_count)]
    
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    
    print(f"   📊 Statistiques:")
    print(f"      Total: {total} images")
    print(f"      Nombre de classes: {len(distribution)}")
    print(f"      Moyenne par classe: {total / len(distribution):.0f}")
    
    print(f"\n   ⚖️ Déséquilibre:")
    print(f"      Classe majoritaire: {max_class} ({max_count} images)")
    print(f"      Classe minoritaire: {min_class} ({min_count} images)")
    print(f"      Ratio: {imbalance_ratio:.1f}:1")
    
    if imbalance_ratio > 10:
        print(f"      ⚠️ DÉSÉQUILIBRE EXTRÊME!")
    elif imbalance_ratio > 5:
        print(f"      ⚠️ DÉSÉQUILIBRE ÉLEVÉ")
    elif imbalance_ratio > 2:
        print(f"      ⚠️ DÉSÉQUILIBRE MODÉRÉ")
    else:
        print(f"      ✅ Dataset relativement équilibré")
    
    print(f"\n   📊 Distribution par classe:")
    for class_name, count in sorted(distribution.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / total) * 100
        print(f"      {class_name}: {count} ({percentage:.1f}%)")
    
    return imbalance_ratio

train_imbalance = analyze_imbalance(train_distribution, "TRAIN SET")
test_imbalance = analyze_imbalance(test_distribution, "TEST SET")

# ============================================================
# 5. VISUALISATION DE LA DISTRIBUTION
# ============================================================

print("\n" + "="*70)
print("📊 VISUALISATION DE LA DISTRIBUTION")
print("="*70)

def plot_distribution(train_dist, test_dist):
    """Visualise la distribution des classes"""
    
    if not train_dist and not test_dist:
        print("Pas de données à visualiser")
        return
    
    fig, axes = plt.subplots(1, 2 if test_dist else 1, figsize=(15, 6))
    
    if test_dist:
        ax1, ax2 = axes
    else:
        ax1 = axes
        ax2 = None
    
    # Train distribution
    if train_dist:
        classes = list(train_dist.keys())
        counts = list(train_dist.values())
        colors = sns.color_palette("husl", len(classes))
        
        bars = ax1.bar(classes, counts, color=colors, edgecolor='black', linewidth=1.5)
        ax1.set_title('Distribution TRAIN SET', fontsize=16, fontweight='bold')
        ax1.set_xlabel('Classe', fontsize=12)
        ax1.set_ylabel('Nombre d\'images', fontsize=12)
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(axis='y', alpha=0.3)
        
        # Ajouter les valeurs sur les barres
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Test distribution
    if test_dist and ax2 is not None:
        classes = list(test_dist.keys())
        counts = list(test_dist.values())
        colors = sns.color_palette("husl", len(classes))
        
        bars = ax2.bar(classes, counts, color=colors, edgecolor='black', linewidth=1.5)
        ax2.set_title('Distribution TEST SET', fontsize=16, fontweight='bold')
        ax2.set_xlabel('Classe', fontsize=12)
        ax2.set_ylabel('Nombre d\'images', fontsize=12)
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(axis='y', alpha=0.3)
        
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../figures/distribution_classes.png', dpi=300, bbox_inches='tight')
    print("✅ Graphique sauvegardé: distribution_classes.png")
    plt.show()

plot_distribution(train_distribution, test_distribution)

# ============================================================
# 6. ANALYSE DES DIMENSIONS DES IMAGES
# ============================================================

print("\n" + "="*70)
print("📏 ANALYSE DES DIMENSIONS DES IMAGES")
print("="*70)

def analyze_image_dimensions(data_dir, max_samples=200):
    """Analyse les dimensions des images"""
    
    if not os.path.exists(data_dir):
        print("Dossier introuvable")
        return
    
    widths = []
    heights = []
    aspect_ratios = []
    
    print(f"Analyse de {max_samples} images...")
    
    classes = [d for d in os.listdir(data_dir) 
              if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
    
    sample_count = 0
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        for img_name in images[:max_samples // len(classes)]:
            try:
                img_path = os.path.join(class_dir, img_name)
                with Image.open(img_path) as img:
                    w, h = img.size
                    widths.append(w)
                    heights.append(h)
                    aspect_ratios.append(w / h)
                    sample_count += 1
            except Exception as e:
                pass
            
            if sample_count >= max_samples:
                break
        
        if sample_count >= max_samples:
            break
    
    if not widths:
        print("Aucune image analysée")
        return
    
    print(f"\n📊 Résultats ({sample_count} images analysées):")
    print(f"   Largeur moyenne: {np.mean(widths):.0f} px (min: {min(widths)}, max: {max(widths)})")
    print(f"   Hauteur moyenne: {np.mean(heights):.0f} px (min: {min(heights)}, max: {max(heights)})")
    print(f"   Ratio d'aspect moyen: {np.mean(aspect_ratios):.2f}")
    
    # Vérifier si redimensionnement nécessaire
    unique_sizes = set(zip(widths, heights))
    print(f"\n   Tailles uniques: {len(unique_sizes)}")
    
    if len(unique_sizes) > 1:
        print(f"   ⚠️ Images de tailles différentes → Redimensionnement NÉCESSAIRE")
        print(f"   ✅ Taille recommandée: {CONFIG['img_size']}x{CONFIG['img_size']} px")
    else:
        print(f"   ✅ Toutes les images ont la même taille")

analyze_image_dimensions(TRAIN_DIR)

# ============================================================
# 7. VISUALISATION D'EXEMPLES PAR CLASSE
# ============================================================

print("\n" + "="*70)
print("🖼️ VISUALISATION D'EXEMPLES PAR CLASSE")
print("="*70)

def visualize_samples_per_class(data_dir, samples_per_class=3):
    """Affiche des exemples d'images par classe"""
    
    if not os.path.exists(data_dir):
        print("Dossier introuvable")
        return
    
    classes = [d for d in os.listdir(data_dir) 
              if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
    classes.sort()
    
    if not classes:
        print("Aucune classe trouvée")
        return
    
    fig, axes = plt.subplots(len(classes), samples_per_class, 
                            figsize=(samples_per_class * 4, len(classes) * 3))
    
    if len(classes) == 1:
        axes = [axes]
    
    fig.suptitle('EXEMPLES D\'IMAGES PAR CLASSE', fontsize=16, fontweight='bold')
    
    for i, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        for j in range(samples_per_class):
            ax = axes[i][j] if len(classes) > 1 else axes[j]
            
            if j < len(images):
                img_path = os.path.join(class_dir, images[j])
                try:
                    img = Image.open(img_path)
                    ax.imshow(img)
                    ax.set_title(f'{class_name}\n{img.size[0]}x{img.size[1]} px', 
                               fontsize=10)
                except Exception as e:
                    ax.text(0.5, 0.5, 'Erreur\nchargement', 
                           ha='center', va='center')
            else:
                ax.text(0.5, 0.5, 'Pas d\'image', ha='center', va='center')
            
            ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('../figures/exemples_par_classe.png', dpi=300, bbox_inches='tight')
    print("✅ Graphique sauvegardé: exemples_par_classe.png")
    plt.show()

visualize_samples_per_class(TRAIN_DIR, samples_per_class=4)

# ============================================================
# 8. DÉFINITION DES TRANSFORMATIONS
# ============================================================

print("\n" + "="*70)
print("🎨 DÉFINITION DES TRANSFORMATIONS")
print("="*70)

# Transformations pour l'entraînement (avec augmentation intensive)
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transformations pour le test (sans augmentation)
test_transform = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Transformations définies:")
print(f"   Train: Resize + 6 augmentations + Normalize")
print(f"   Test: Resize + Normalize")

# ============================================================
# 9. TEST DES TRANSFORMATIONS
# ============================================================

print("\n" + "="*70)
print("🧪 TEST DES TRANSFORMATIONS")
print("="*70)

def test_transformations(data_dir):
    """Teste les transformations sur une image réelle"""
    
    if not os.path.exists(data_dir):
        print("Dossier introuvable")
        return
    
    # Trouver une image
    classes = [d for d in os.listdir(data_dir) 
              if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
    
    if not classes:
        print("Aucune classe trouvée")
        return
    
    class_dir = os.path.join(data_dir, classes[0])
    images = [f for f in os.listdir(class_dir) 
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    if not images:
        print("Aucune image trouvée")
        return
    
    img_path = os.path.join(class_dir, images[0])
    
    try:
        # Charger l'image
        original_img = Image.open(img_path).convert('RGB')
        
        # Appliquer les transformations
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Original
        axes[0].imshow(original_img)
        axes[0].set_title(f'Original\n{original_img.size[0]}x{original_img.size[1]} px', 
                         fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        # Après transformation train
        train_img = train_transform(original_img)
        # Dénormaliser pour affichage
        train_img_display = train_img.clone()
        for t, m, s in zip(train_img_display, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]):
            t.mul_(s).add_(m)
        train_img_display = torch.clamp(train_img_display, 0, 1)
        axes[1].imshow(train_img_display.permute(1, 2, 0))
        axes[1].set_title(f'Train Transform\n{CONFIG["img_size"]}x{CONFIG["img_size"]} px\n(avec augmentation)', 
                         fontsize=12, fontweight='bold')
        axes[1].axis('off')
        
        # Après transformation test
        test_img = test_transform(original_img)
        # Dénormaliser pour affichage
        test_img_display = test_img.clone()
        for t, m, s in zip(test_img_display, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]):
            t.mul_(s).add_(m)
        test_img_display = torch.clamp(test_img_display, 0, 1)
        axes[2].imshow(test_img_display.permute(1, 2, 0))
        axes[2].set_title(f'Test Transform\n{CONFIG["img_size"]}x{CONFIG["img_size"]} px\n(sans augmentation)', 
                         fontsize=12, fontweight='bold')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.savefig('../figures/test_transformations.png', dpi=300, bbox_inches='tight')
        print("✅ Graphique sauvegardé: test_transformations.png")
        plt.show()
        
        print(f"\n✅ Test réussi:")
        print(f"   Image originale: {original_img.size}")
        print(f"   Après transformation: {train_img.shape}")
        print(f"   Range des pixels: [{train_img.min():.2f}, {train_img.max():.2f}]")
        
    except Exception as e:
        print(f"❌ Erreur: {e}")

test_transformations(TRAIN_DIR)

# ============================================================
# 10. CALCUL DES CLASS WEIGHTS
# ============================================================

print("\n" + "="*70)
print("⚖️ CALCUL DES CLASS WEIGHTS")
print("="*70)

def calculate_class_weights(distribution):
    """Calcule les class weights avec Effective Number of Samples"""
    
    if not distribution:
        print("Pas de données")
        return None
    
    # Méthode Effective Number of Samples (meilleure pour déséquilibre extrême)
    beta = 0.9999
    class_weights = {}
    
    for class_name, count in distribution.items():
        effective_num = 1.0 - np.power(beta, count)
        class_weights[class_name] = (1.0 - beta) / effective_num
    
    # Normaliser
    total_weight = sum(class_weights.values())
    for class_name in class_weights:
        class_weights[class_name] = class_weights[class_name] / total_weight * len(class_weights)
    
    print("📊 Class Weights (Effective Number of Samples):")
    for class_name, weight in sorted(class_weights.items(), key=lambda x: x[1], reverse=True):
        print(f"   {class_name}: {weight:.2f}")
    
    return class_weights

train_class_weights = calculate_class_weights(train_distribution)

# ============================================================
# 11. RAPPORT FINAL
# ============================================================

print("\n" + "="*70)
print("📋 RAPPORT FINAL D'EXPLORATION")
print("="*70)

print(f"\n🏷️ CLASSES:")
if train_distribution:
    for i, (class_name, count) in enumerate(sorted(train_distribution.items()), 1):
        test_count = test_distribution.get(class_name, 0) if test_distribution else 0
        print(f"   {i}. {class_name} (Train: {count}, Test: {test_count})")

print(f"\n📊 STATISTIQUES:")
total_train = sum(train_distribution.values()) if train_distribution else 0
total_test = sum(test_distribution.values()) if test_distribution else 0
print(f"   Train: {total_train} images")
print(f"   Test: {total_test} images")
print(f"   Total: {total_train + total_test} images")

if train_imbalance:
    print(f"\n⚖️ DÉSÉQUILIBRE:")
    print(f"   Ratio: {train_imbalance:.1f}:1")
    if train_imbalance > 10:
        print(f"   ⚠️ EXTRÊME → Focal Loss + Class Weights + WeightedSampler OBLIGATOIRES")
    elif train_imbalance > 5:
        print(f"   ⚠️ ÉLEVÉ → Class Weights + WeightedSampler recommandés")

print(f"\n🎯 RECOMMANDATIONS POUR L'ENTRAÎNEMENT:")
print(f"   1. Utiliser WeightedRandomSampler pour équilibrer les batches")
print(f"   2. Utiliser Focal Loss au lieu de CrossEntropyLoss")
print(f"   3. Appliquer les Class Weights calculés")
print(f"   4. Monitorer la Balanced Accuracy plutôt que l'Accuracy")
print(f"   5. Data Augmentation intensive (déjà configurée)")

if total_test == 0:
    print(f"\n⚠️ TEST SET VIDE:")
    print(f"   → Créer un split train/validation (80/20) lors de l'entraînement")
    print(f"   → Utiliser sklearn.model_selection.train_test_split avec stratify")

print(f"\n✅ CONFIGURATION PRÊTE:")
print(f"   Taille des images: {CONFIG['img_size']}x{CONFIG['img_size']} px")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Transformations: Définies")
print(f"   Class Weights: Calculés")

print(f"\n📁 FICHIERS GÉNÉRÉS:")
print(f"   - distribution_classes.png")
print(f"   - exemples_par_classe.png")
print(f"   - test_transformations.png")

print("\n" + "="*70)
print("✨ EXPLORATION TERMINÉE - PRÊT POUR L'ENTRAÎNEMENT!")
print("="*70)

print(f"\n💡 PROCHAINE ÉTAPE:")
print(f"   Dans un nouveau notebook, utilisez:")
print(f"   - Les transformations définies ici")
print(f"   - Les class weights calculés")
print(f"   - WeightedRandomSampler + Focal Loss")
print(f"   - Split train/val car test est vide")

In [ ]:
# ============================================================
# 7. RÉSOLUTION DU DÉSÉQUILIBRE ET PRÉTRAITEMENT DES IMAGES
# ============================================================
# ============================================================
# 7.1 DATASET AVEC PRÉTRAITEMENT RÉEL
# ============================================================

class AlzheimerDataset(Dataset):
    def __init__(self, data_dir, transform=None, balance_classes=False):
        self.data_dir = data_dir
        self.transform = transform
        self.balance_classes = balance_classes
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        
        # Charger les classes
        classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
        classes.sort()
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        self.idx_to_class = {idx: cls for idx, cls in enumerate(classes)}
        
        # Charger les images avec leurs labels
        for class_name in classes:
            class_dir = os.path.join(data_dir, class_name)
            if os.path.exists(class_dir):
                images = [f for f in os.listdir(class_dir) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                for img_name in images:
                    self.images.append(os.path.join(class_dir, img_name))
                    self.labels.append(self.class_to_idx[class_name])
        
        # Calculer les weights pour le balancing
        if self.balance_classes and self.labels:
            self.class_weights = self._calculate_class_weights()
            self.sample_weights = [self.class_weights[label] for label in self.labels]
    
    def _calculate_class_weights(self):
        """Calcule les poids de classe pour équilibrer le dataset"""
        class_counts = Counter(self.labels)
        total_samples = len(self.labels)
        num_classes = len(class_counts)
        
        class_weights = {}
        for class_idx, count in class_counts.items():
            # Formule: total_samples / (num_classes * count)
            class_weights[class_idx] = total_samples / (num_classes * count)
        
        return class_weights
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Chargement et transformation de l'image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)  # REDIMENSIONNEMENT APPLIQUÉ ICI
        
        return image, label

# ============================================================
# 7.2 CRÉATION DES DATALOADERS AVEC RÉSOLUTION DU DÉSÉQUILIBRE
# ============================================================

def create_balanced_dataloaders():
    """Crée les dataloaders avec gestion du déséquilibre"""
    
    print("🔄 Création des dataloaders équilibrés...")
    
    # Créer les datasets
    train_dataset = AlzheimerDataset(
        CONFIG['train_dir'], 
        transform=train_transform,  # REDIMENSIONNEMENT + AUGMENTATION
        balance_classes=True  # Active le calcul des weights
    )
    
    test_dataset = AlzheimerDataset(
        CONFIG['test_dir'],
        transform=test_transform  # REDIMENSIONNEMENT SEULEMENT
    )
    
    print(f"📦 Datasets créés:")
    print(f"   Train: {len(train_dataset)} images, {len(train_dataset.class_to_idx)} classes")
    print(f"   Test:  {len(test_dataset)} images, {len(test_dataset.class_to_idx)} classes")
    
    # Afficher la distribution originale
    train_counts = Counter(train_dataset.labels)
    print(f"\n📊 Distribution originale (Train):")
    for class_idx, count in train_counts.items():
        class_name = train_dataset.idx_to_class[class_idx]
        print(f"   {class_name}: {count} images")
    
    # Créer le WeightedRandomSampler pour équilibrer le train
    sampler = WeightedRandomSampler(
        train_dataset.sample_weights,
        num_samples=len(train_dataset.sample_weights),
        replacement=True
    )
    
    # Créer les dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        sampler=sampler,  # Utilise le sampler pour équilibrer
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    
    # Vérifier l'équilibrage
    print(f"\n✅ Dataloaders créés avec WeightedRandomSampler")
    print(f"   Chaque batch sera équilibré automatiquement")
    
    return train_loader, test_loader, train_dataset, test_dataset

# ============================================================
# 7.3 VÉRIFICATION DE L'ÉQUILIBRAGE
# ============================================================

def verify_balancing(train_loader, train_dataset):
    """Vérifie que le dataloader est bien équilibré"""
    
    print("\n🔍 Vérification de l'équilibrage...")
    
    # Analyser la distribution dans les batches
    batch_distributions = []
    
    for i, (images, labels) in enumerate(train_loader):
        if i >= 3:  
            break
        
        unique, counts = torch.unique(labels, return_counts=True)
        batch_dist = {train_dataset.idx_to_class[idx.item()]: count.item() 
                    for idx, count in zip(unique, counts)}
        batch_distributions.append(batch_dist)
    
    print("📊 Distribution dans les batches (après équilibrage):")
    for i, dist in enumerate(batch_distributions):
        print(f"   Batch {i+1}: {dist}")
    
    # Calculer l'équilibre moyen
    avg_dist = {}
    for dist in batch_distributions:
        for cls, count in dist.items():
            avg_dist[cls] = avg_dist.get(cls, 0) + count
    
    for cls in avg_dist:
        avg_dist[cls] = avg_dist[cls] // len(batch_distributions)
    
    print(f"   Distribution moyenne par batch: {avg_dist}")

# ============================================================
# 7.4 CALCUL DES CLASS WEIGHTS POUR LA LOSS FUNCTION
# ============================================================

def calculate_class_weights_for_loss(train_dataset):
    """Calcule les poids de classe pour la loss function"""
    
    print("\n⚖️ Calcul des Class Weights pour la loss function...")
    
    if not hasattr(train_dataset, 'class_weights'):
        # Calculer manuellement si pas déjà fait
        class_counts = Counter(train_dataset.labels)
        total_samples = len(train_dataset.labels)
        num_classes = len(class_counts)
        
        class_weights = {}
        for class_idx, count in class_counts.items():
            class_weights[class_idx] = total_samples / (num_classes * count)
        
        train_dataset.class_weights = class_weights
    
    # Convertir en tensor pour PyTorch
    weights_tensor = torch.tensor([train_dataset.class_weights[i] 
                                for i in range(len(train_dataset.class_weights))],
                                dtype=torch.float32)
    
    print("📊 Class Weights calculés:")
    for class_idx, weight in train_dataset.class_weights.items():
        class_name = train_dataset.idx_to_class[class_idx]
        print(f"   {class_name}: {weight:.2f}")
    
    return weights_tensor

# ============================================================
# 7.5 TEST DES TRANSFORMATIONS SUR DES IMAGES RÉELLES
# ============================================================

def test_transformations():
    """Teste les transformations sur des images réelles"""
    
    print("\n🧪 Test des transformations...")
    
    # Créer un dataset de test sans balancement
    test_dataset = AlzheimerDataset(CONFIG['train_dir'], transform=basic_transform)
    
    if len(test_dataset) == 0:
        print("❌ Aucune image trouvée pour le test")
        return
    
    # Prendre une image de chaque classe
    fig, axes = plt.subplots(3, 4, figsize=(15, 10))
    fig.suptitle('TEST DES TRANSFORMATIONS - AVANT/APRÈS REDIMENSIONNEMENT', 
                 fontsize=16, fontweight='bold')
    
    class_images = {}
    
    # Trouver une image par classe
    for img_path, label in zip(test_dataset.images, test_dataset.labels):
        class_name = test_dataset.idx_to_class[label]
        if class_name not in class_images:
            class_images[class_name] = img_path
        if len(class_images) == len(test_dataset.class_to_idx):
            break
    
    # Afficher les transformations
    for i, (class_name, img_path) in enumerate(class_images.items()):
        if i >= 3:  # Limiter à 3 classes pour l'affichage
            break
            
        # Image originale
        original_img = Image.open(img_path).convert('RGB')
        axes[i, 0].imshow(original_img)
        axes[i, 0].set_title(f'{class_name}\nOriginal\n{original_img.size}', fontsize=10)
        axes[i, 0].axis('off')
        
        # Après basic transform (redimensionnement)
        basic_img = basic_transform(original_img)
        axes[i, 1].imshow(basic_img.permute(1, 2, 0))
        axes[i, 1].set_title(f'Redimensionné\n{basic_img.shape}', fontsize=10)
        axes[i, 1].axis('off')
        
        # Après train transform
        train_img = train_transform(original_img)
        axes[i, 2].imshow(train_img.permute(1, 2, 0))
        axes[i, 2].set_title(f'Augmenté\n{train_img.shape}', fontsize=10)
        axes[i, 2].axis('off')
        
        # Après test transform
        test_img = test_transform(original_img)
        axes[i, 3].imshow(test_img.permute(1, 2, 0))
        axes[i, 3].set_title(f'Test\n{test_img.shape}', fontsize=10)
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('transformations_test.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# ============================================================
# 8. SAUVEGARDE DES IMAGES PRÉTRAITÉES
# ============================================================

import os
import torch
from PIL import Image
import shutil
from tqdm import tqdm

# Chemin pour sauvegarder les images prétraitées
PRETRAITE_DIR = "C:/Users/angej/Desktop/MultimodalAI/Alzheimer/data/Pretraite"
PRETRAITE_TRAIN_DIR = os.path.join(PRETRAITE_DIR, "train")
PRETRAITE_TEST_DIR = os.path.join(PRETRAITE_DIR, "test")

def save_processed_images():
    """Sauvegarde toutes les images prétraitées"""
    
    print("\n💾 SAUVEGARDE DES IMAGES PRÉTRAITÉES...")
    print(f"📍 Dossier de destination: {PRETRAITE_DIR}")
    
    # Créer les dossiers
    os.makedirs(PRETRAITE_TRAIN_DIR, exist_ok=True)
    os.makedirs(PRETRAITE_TEST_DIR, exist_ok=True)
    
    # Sauvegarde du train set
    print("\n📁 Sauvegarde du train set...")
    train_saved = 0
    train_dataset = AlzheimerDataset(CONFIG['train_dir'], transform=train_transform)
    
    for class_name in train_dataset.class_to_idx.keys():
        class_dir = os.path.join(PRETRAITE_TRAIN_DIR, class_name)
        os.makedirs(class_dir, exist_ok=True)
    
    for i, (img_path, label) in enumerate(tqdm(zip(train_dataset.images, train_dataset.labels), 
                                            total=len(train_dataset), desc="Train")):
        try:
            # Charger et transformer l'image
            image = Image.open(img_path).convert('RGB')
            processed_image = train_transform(image)
            
            # Convertir le tensor en image PIL
            processed_image = transforms.ToPILImage()(processed_image)
            
            # Sauvegarder
            class_name = train_dataset.idx_to_class[label]
            filename = os.path.basename(img_path)
            save_path = os.path.join(PRETRAITE_TRAIN_DIR, class_name, filename)
            processed_image.save(save_path, quality=95)
            
            train_saved += 1
            
        except Exception as e:
            print(f"❌ Erreur avec {img_path}: {e}")
    
    # Sauvegarde du test set  
    print("\n📁 Sauvegarde du test set...")
    test_saved = 0
    test_dataset = AlzheimerDataset(CONFIG['test_dir'], transform=test_transform)
    
    for class_name in test_dataset.class_to_idx.keys():
        class_dir = os.path.join(PRETRAITE_TEST_DIR, class_name)
        os.makedirs(class_dir, exist_ok=True)
    
    for i, (img_path, label) in enumerate(tqdm(zip(test_dataset.images, test_dataset.labels), 
                                            total=len(test_dataset), desc="Test")):
        try:
            # Charger et transformer l'image
            image = Image.open(img_path).convert('RGB')
            processed_image = test_transform(image)
            
            # Convertir le tensor en image PIL
            processed_image = transforms.ToPILImage()(processed_image)
            
            # Sauvegarder
            class_name = test_dataset.idx_to_class[label]
            filename = os.path.basename(img_path)
            save_path = os.path.join(PRETRAITE_TEST_DIR, class_name, filename)
            processed_image.save(save_path, quality=95)
            
            test_saved += 1
            
        except Exception as e:
            print(f"❌ Erreur avec {img_path}: {e}")
    
    return train_saved, test_saved

# ============================================================
# 9. VÉRIFICATION DES IMAGES SAUVEGARDÉES
# ============================================================

def verify_saved_images():
    """Vérifie que les images ont bien été sauvegardées"""
    
    print("\n🔍 VÉRIFICATION DES IMAGES SAUVEGARDÉES...")
    
    if not os.path.exists(PRETRAITE_DIR):
        print("❌ Dossier de sauvegarde introuvable")
        return
    
    print(f"📁 Structure créée:")
    print(f"   {PRETRAITE_DIR}")
    
    # Vérifier train
    if os.path.exists(PRETRAITE_TRAIN_DIR):
        train_classes = [d for d in os.listdir(PRETRAITE_TRAIN_DIR) 
                        if os.path.isdir(os.path.join(PRETRAITE_TRAIN_DIR, d))]
        print(f"\n📊 Train sauvegardé ({len(train_classes)} classes):")
        for class_name in train_classes:
            class_dir = os.path.join(PRETRAITE_TRAIN_DIR, class_name)
            images = [f for f in os.listdir(class_dir) 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            print(f"   {class_name}: {len(images)} images")
    
    # Vérifier test
    if os.path.exists(PRETRAITE_TEST_DIR):
        test_classes = [d for d in os.listdir(PRETRAITE_TEST_DIR) 
                    if os.path.isdir(os.path.join(PRETRAITE_TEST_DIR, d))]
        print(f"\n📊 Test sauvegardé ({len(test_classes)} classes):")
        for class_name in test_classes:
            class_dir = os.path.join(PRETRAITE_TEST_DIR, class_name)
            images = [f for f in os.listdir(class_dir) 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            print(f"   {class_name}: {len(images)} images")

# ============================================================
# 10. COMPARAISON AVANT/APRÈS PRÉTRAITEMENT
# ============================================================

def compare_before_after():
    """Montre la différence avant/après prétraitement"""
    
    print("\n🔍 COMPARAISON AVANT/APRÈS PRÉTRAITEMENT...")
    
    # Prendre quelques images originales
    original_dataset = AlzheimerDataset(CONFIG['train_dir'], transform=basic_transform)
    
    if len(original_dataset) == 0:
        print("❌ Aucune image trouvée pour la comparaison")
        return
    
    # Trouver des images de différentes classes
    class_images = {}
    for img_path, label in zip(original_dataset.images, original_dataset.labels):
        class_name = original_dataset.idx_to_class[label]
        if class_name not in class_images:
            class_images[class_name] = img_path
        if len(class_images) >= 3:
            break
    
    # Créer la visualisation
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle('COMPARAISON AVANT/APRÈS PRÉTRAITEMENT', fontsize=16, fontweight='bold')
    
    for i, (class_name, img_path) in enumerate(class_images.items()):
        # Image originale
        original_img = Image.open(img_path).convert('RGB')
        axes[i, 0].imshow(original_img)
        axes[i, 0].set_title(f'{class_name}\nAVANT\n{original_img.size}', fontsize=10)
        axes[i, 0].axis('off')
        
        # Après prétraitement (train)
        train_img = train_transform(original_img)
        train_pil = transforms.ToPILImage()(train_img)
        axes[i, 1].imshow(train_pil)
        axes[i, 1].set_title(f'APRÈS (Train)\n{train_img.shape}', fontsize=10)
        axes[i, 1].axis('off')
        
        # Après prétraitement (test)
        test_img = test_transform(original_img)
        test_pil = transforms.ToPILImage()(test_img)
        axes[i, 2].imshow(test_pil)
        axes[i, 2].set_title(f'APRÈS (Test)\n{test_img.shape}', fontsize=10)
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('comparaison_pretraitement.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================================
# 11. EXÉCUTION DE LA SAUVEGARDE
# ============================================================

print("\n" + "="*70)
print("💾 DÉMARRAGE DE LA SAUVEGARDE DES IMAGES PRÉTRAITÉES")
print("="*70)

# 1. Sauvegarder toutes les images prétraitées
train_saved, test_saved = save_processed_images()

# 2. Vérifier la sauvegarde
verify_saved_images()

# 3. Montrer la comparaison
compare_before_after()

# ============================================================
# 12. RAPPORT FINAL DE SAUVEGARDE
# ============================================================

print("\n" + "="*70)
print("✅ SAUVEGARDE TERMINÉE - RAPPORT FINAL")
print("="*70)

print(f"\n📊 STATISTIQUES DE SAUVEGARDE:")
print(f"   ✅ Images train sauvegardées: {train_saved}")
print(f"   ✅ Images test sauvegardées: {test_saved}")
print(f"   ✅ Total images: {train_saved + test_saved}")

print(f"\n📁 DOSSIERS CRÉÉS:")
print(f"   {PRETRAITE_TRAIN_DIR}")
print(f"   {PRETRAITE_TEST_DIR}")

print(f"\n🎯 CARACTÉRISTIQUES DES IMAGES SAUVEGARDÉES:")
print(f"   • Taille: {CONFIG['img_size']}x{CONFIG['img_size']} pixels")
print(f"   • Format: JPEG (qualité 95%)")
print(f"   • Train: Augmentation appliquée")
print(f"   • Test: Redimensionnement + Normalisation seulement")

print(f"\n🔧 POUR UTILISER LES NOUVELLES IMAGES:")
print(f"   Remplacez dans votre code:")
print(f"   AVANT: {CONFIG['train_dir']}")
print(f"   APRÈS: {PRETRAITE_TRAIN_DIR}")
print(f"   ")
print(f"   AVANT: {CONFIG['test_dir']}")
print(f"   APRÈS: {PRETRAITE_TEST_DIR}")

print(f"\n📝 EXEMPLE D'UTILISATION:")
print(f"   # Avec les nouvelles images prétraitées")
print(f"   train_dataset = AlzheimerDataset('{PRETRAITE_TRAIN_DIR}', transform=basic_transform)")
print(f"   test_dataset = AlzheimerDataset('{PRETRAITE_TEST_DIR}', transform=basic_transform)")

print(f"\n💡 AVANTAGE:")
print(f"   Les images sont déjà prétraitées → chargement plus rapide")
print(f"   Pas besoin de transformations complexes pendant l'entraînement")

print("\n" + "="*70)
print("🎉 TOUTES LES IMAGES SONT MAINTENANT PRÉTRAITÉES ET SAUVEGARDÉES!")
print("="*70)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from torchvision import transforms
from collections import Counter
import json
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURATION ET CHARGEMENT
# ============================================================

print("\n" + "="*70)
print("📊 VISUALISATION COMPARÉE - DONNÉES BRUTES VS PRÉTRAITÉES")
print("="*70)

# Chemins
BASE_DIR = "C:/Users/angej/Desktop/MultimodalAI/Alzheimer/data"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
PRETRAITE_DIR = os.path.join(BASE_DIR, "pretraite")

RAW_TRAIN_DIR = os.path.join(PROCESSED_DIR, "train")
PREP_TRAIN_DIR = os.path.join(PRETRAITE_DIR, "train")
CONFIG_FILE = os.path.join(PRETRAITE_DIR, "dataset_config.json")

print(f"\n📁 Chemins:")
print(f"   Données brutes: {RAW_TRAIN_DIR}")
print(f"   Données prétraitées: {PREP_TRAIN_DIR}")

# Charger la configuration
if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        config = json.load(f)
    print(f"\n✅ Configuration chargée")
else:
    print(f"\n⚠️ Configuration non trouvée")
    config = None

# ============================================================
# 2. ANALYSE DES DIMENSIONS - AVANT/APRÈS
# ============================================================

print("\n" + "="*70)
print("📏 ANALYSE DES DIMENSIONS - AVANT/APRÈS")
print("="*70)

def analyze_dimensions(data_dir, max_samples=100):
    """Analyse les dimensions des images"""
    
    widths = []
    heights = []
    sizes_mb = []
    
    classes = [d for d in os.listdir(data_dir) 
              if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
    
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        for img_name in images[:max_samples // len(classes)]:
            try:
                img_path = os.path.join(class_dir, img_name)
                with Image.open(img_path) as img:
                    w, h = img.size
                    widths.append(w)
                    heights.append(h)
                    
                    # Taille du fichier
                    size_mb = os.path.getsize(img_path) / (1024 * 1024)
                    sizes_mb.append(size_mb)
            except:
                pass
    
    return widths, heights, sizes_mb

# Analyser les données brutes
print("\n🔍 Analyse des données BRUTES...")
raw_widths, raw_heights, raw_sizes = analyze_dimensions(RAW_TRAIN_DIR)

if raw_widths:
    print(f"   Échantillon analysé: {len(raw_widths)} images")
    print(f"   Largeur: min={min(raw_widths)}, max={max(raw_widths)}, moy={np.mean(raw_widths):.0f}")
    print(f"   Hauteur: min={min(raw_heights)}, max={max(raw_heights)}, moy={np.mean(raw_heights):.0f}")
    print(f"   Taille fichier: min={min(raw_sizes):.3f}MB, max={max(raw_sizes):.3f}MB, moy={np.mean(raw_sizes):.3f}MB")
    print(f"   Tailles uniques: {len(set(zip(raw_widths, raw_heights)))}")

# Analyser les données prétraitées
print("\n🔍 Analyse des données PRÉTRAITÉES...")
prep_widths, prep_heights, prep_sizes = analyze_dimensions(PREP_TRAIN_DIR)

if prep_widths:
    print(f"   Échantillon analysé: {len(prep_widths)} images")
    print(f"   Largeur: min={min(prep_widths)}, max={max(prep_widths)}, moy={np.mean(prep_widths):.0f}")
    print(f"   Hauteur: min={min(prep_heights)}, max={max(prep_heights)}, moy={np.mean(prep_heights):.0f}")
    print(f"   Taille fichier: min={min(prep_sizes):.3f}MB, max={max(prep_sizes):.3f}MB, moy={np.mean(prep_sizes):.3f}MB")
    print(f"   Tailles uniques: {len(set(zip(prep_widths, prep_heights)))}")

# ============================================================
# 3. GRAPHIQUE COMPARATIF DES DIMENSIONS
# ============================================================

print("\n" + "="*70)
print("📊 GRAPHIQUES COMPARATIFS DES DIMENSIONS")
print("="*70)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('COMPARAISON DIMENSIONS - BRUTES vs PRÉTRAITÉES', 
             fontsize=16, fontweight='bold')

# Distribution des largeurs
axes[0, 0].hist(raw_widths, bins=30, alpha=0.7, label='Brutes', color='red', edgecolor='black')
axes[0, 0].axvline(np.mean(raw_widths), color='red', linestyle='--', linewidth=2, label=f'Moy: {np.mean(raw_widths):.0f}')
axes[0, 0].set_title('Distribution des LARGEURS (Brutes)', fontweight='bold')
axes[0, 0].set_xlabel('Largeur (pixels)')
axes[0, 0].set_ylabel('Fréquence')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[1, 0].hist(prep_widths, bins=30, alpha=0.7, label='Prétraitées', color='green', edgecolor='black')
axes[1, 0].axvline(np.mean(prep_widths), color='green', linestyle='--', linewidth=2, label=f'Moy: {np.mean(prep_widths):.0f}')
axes[1, 0].set_title('Distribution des LARGEURS (Prétraitées)', fontweight='bold')
axes[1, 0].set_xlabel('Largeur (pixels)')
axes[1, 0].set_ylabel('Fréquence')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Distribution des hauteurs
axes[0, 1].hist(raw_heights, bins=30, alpha=0.7, label='Brutes', color='red', edgecolor='black')
axes[0, 1].axvline(np.mean(raw_heights), color='red', linestyle='--', linewidth=2, label=f'Moy: {np.mean(raw_heights):.0f}')
axes[0, 1].set_title('Distribution des HAUTEURS (Brutes)', fontweight='bold')
axes[0, 1].set_xlabel('Hauteur (pixels)')
axes[0, 1].set_ylabel('Fréquence')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 1].hist(prep_heights, bins=30, alpha=0.7, label='Prétraitées', color='green', edgecolor='black')
axes[1, 1].axvline(np.mean(prep_heights), color='green', linestyle='--', linewidth=2, label=f'Moy: {np.mean(prep_heights):.0f}')
axes[1, 1].set_title('Distribution des HAUTEURS (Prétraitées)', fontweight='bold')
axes[1, 1].set_xlabel('Hauteur (pixels)')
axes[1, 1].set_ylabel('Fréquence')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

# Tailles des fichiers
axes[0, 2].hist(raw_sizes, bins=30, alpha=0.7, label='Brutes', color='red', edgecolor='black')
axes[0, 2].axvline(np.mean(raw_sizes), color='red', linestyle='--', linewidth=2, label=f'Moy: {np.mean(raw_sizes):.3f}MB')
axes[0, 2].set_title('TAILLE DES FICHIERS (Brutes)', fontweight='bold')
axes[0, 2].set_xlabel('Taille (MB)')
axes[0, 2].set_ylabel('Fréquence')
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

axes[1, 2].hist(prep_sizes, bins=30, alpha=0.7, label='Prétraitées', color='green', edgecolor='black')
axes[1, 2].axvline(np.mean(prep_sizes), color='green', linestyle='--', linewidth=2, label=f'Moy: {np.mean(prep_sizes):.3f}MB')
axes[1, 2].set_title('TAILLE DES FICHIERS (Prétraitées)', fontweight='bold')
axes[1, 2].set_xlabel('Taille (MB)')
axes[1, 2].set_ylabel('Fréquence')
axes[1, 2].legend()
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('dimensions_comparison.png', dpi=300, bbox_inches='tight')
print("✅ Graphique sauvegardé: dimensions_comparison.png")
plt.show()

# ============================================================
# 4. COMPARAISON VISUELLE - ÉCHANTILLONS PAR CLASSE
# ============================================================

print("\n" + "="*70)
print("🖼️ COMPARAISON VISUELLE - ÉCHANTILLONS PAR CLASSE")
print("="*70)

# Récupérer les classes
classes = [d for d in os.listdir(RAW_TRAIN_DIR) 
          if os.path.isdir(os.path.join(RAW_TRAIN_DIR, d)) and not d.startswith('_')]
classes.sort()

print(f"\n📂 Classes trouvées: {classes}")

# Créer la visualisation
n_classes = len(classes)
n_samples = 3

fig, axes = plt.subplots(n_classes, n_samples * 2, figsize=(n_samples * 6, n_classes * 3))
fig.suptitle('COMPARAISON PAR CLASSE - BRUTES (gauche) vs PRÉTRAITÉES (droite)', 
             fontsize=16, fontweight='bold', y=0.995)

for i, class_name in enumerate(classes):
    # Chemins
    raw_class_dir = os.path.join(RAW_TRAIN_DIR, class_name)
    prep_class_dir = os.path.join(PREP_TRAIN_DIR, class_name)
    
    # Images de la classe
    images = [f for f in os.listdir(raw_class_dir) 
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))][:n_samples]
    
    for j, img_name in enumerate(images):
        # Image brute
        raw_path = os.path.join(raw_class_dir, img_name)
        raw_img = Image.open(raw_path)
        
        ax_raw = axes[i, j*2] if n_classes > 1 else axes[j*2]
        ax_raw.imshow(raw_img)
        ax_raw.set_title(f'{class_name}\nBRUTE\n{raw_img.size[0]}x{raw_img.size[1]}px', 
                        fontsize=9, fontweight='bold', color='red')
        ax_raw.axis('off')
        
        # Image prétraitée
        prep_path = os.path.join(prep_class_dir, img_name)
        if os.path.exists(prep_path):
            prep_img = Image.open(prep_path)
            
            ax_prep = axes[i, j*2+1] if n_classes > 1 else axes[j*2+1]
            ax_prep.imshow(prep_img)
            ax_prep.set_title(f'{class_name}\nPRÉTRAITÉE\n{prep_img.size[0]}x{prep_img.size[1]}px', 
                            fontsize=9, fontweight='bold', color='green')
            ax_prep.axis('off')

plt.tight_layout()
plt.savefig('visual_comparison_by_class.png', dpi=300, bbox_inches='tight')
print("✅ Graphique sauvegardé: visual_comparison_by_class.png")
plt.show()

# ============================================================
# 5. ANALYSE DES INTENSITÉS DE PIXELS
# ============================================================

print("\n" + "="*70)
print("🎨 ANALYSE DES INTENSITÉS DE PIXELS")
print("="*70)

def analyze_pixel_intensities(data_dir, max_samples=50):
    """Analyse les intensités de pixels"""
    
    all_means = []
    all_stds = []
    channel_means = {'R': [], 'G': [], 'B': []}
    
    classes = [d for d in os.listdir(data_dir) 
              if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
    
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        for img_name in images[:max_samples // len(classes)]:
            try:
                img_path = os.path.join(class_dir, img_name)
                img = Image.open(img_path).convert('RGB')
                img_array = np.array(img) / 255.0  # Normaliser 0-1
                
                all_means.append(img_array.mean())
                all_stds.append(img_array.std())
                
                # Par canal
                channel_means['R'].append(img_array[:,:,0].mean())
                channel_means['G'].append(img_array[:,:,1].mean())
                channel_means['B'].append(img_array[:,:,2].mean())
            except:
                pass
    
    return all_means, all_stds, channel_means

# Analyser les intensités
print("\n🔍 Analyse des intensités - BRUTES...")
raw_means, raw_stds, raw_channels = analyze_pixel_intensities(RAW_TRAIN_DIR)

if raw_means:
    print(f"   Intensité moyenne: {np.mean(raw_means):.3f} ± {np.std(raw_means):.3f}")
    print(f"   Écart-type moyen: {np.mean(raw_stds):.3f}")
    print(f"   Canaux RGB:")
    print(f"      R: {np.mean(raw_channels['R']):.3f}")
    print(f"      G: {np.mean(raw_channels['G']):.3f}")
    print(f"      B: {np.mean(raw_channels['B']):.3f}")

print("\n🔍 Analyse des intensités - PRÉTRAITÉES...")
prep_means, prep_stds, prep_channels = analyze_pixel_intensities(PREP_TRAIN_DIR)

if prep_means:
    print(f"   Intensité moyenne: {np.mean(prep_means):.3f} ± {np.std(prep_means):.3f}")
    print(f"   Écart-type moyen: {np.mean(prep_stds):.3f}")
    print(f"   Canaux RGB:")
    print(f"      R: {np.mean(prep_channels['R']):.3f}")
    print(f"      G: {np.mean(prep_channels['G']):.3f}")
    print(f"      B: {np.mean(prep_channels['B']):.3f}")

# Graphiques
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('COMPARAISON DES INTENSITÉS DE PIXELS', fontsize=16, fontweight='bold')

# Intensités moyennes
axes[0, 0].hist(raw_means, bins=30, alpha=0.6, label='Brutes', color='red', edgecolor='black')
axes[0, 0].hist(prep_means, bins=30, alpha=0.6, label='Prétraitées', color='green', edgecolor='black')
axes[0, 0].set_title('Distribution des INTENSITÉS MOYENNES', fontweight='bold')
axes[0, 0].set_xlabel('Intensité moyenne (0-1)')
axes[0, 0].set_ylabel('Fréquence')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Écarts-types
axes[0, 1].hist(raw_stds, bins=30, alpha=0.6, label='Brutes', color='red', edgecolor='black')
axes[0, 1].hist(prep_stds, bins=30, alpha=0.6, label='Prétraitées', color='green', edgecolor='black')
axes[0, 1].set_title('Distribution des ÉCARTS-TYPES', fontweight='bold')
axes[0, 1].set_xlabel('Écart-type')
axes[0, 1].set_ylabel('Fréquence')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Comparaison par canal - Brutes
channels = ['R', 'G', 'B']
colors = ['red', 'green', 'blue']
x = np.arange(len(channels))
width = 0.35

raw_channel_values = [np.mean(raw_channels[c]) for c in channels]
axes[1, 0].bar(x, raw_channel_values, width, color=colors, alpha=0.7, edgecolor='black')
axes[1, 0].set_title('CANAUX RGB - Brutes', fontweight='bold')
axes[1, 0].set_xlabel('Canal')
axes[1, 0].set_ylabel('Intensité moyenne (0-1)')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(channels)
axes[1, 0].grid(axis='y', alpha=0.3)
axes[1, 0].set_ylim([0, 1])

# Comparaison par canal - Prétraitées
prep_channel_values = [np.mean(prep_channels[c]) for c in channels]
axes[1, 1].bar(x, prep_channel_values, width, color=colors, alpha=0.7, edgecolor='black')
axes[1, 1].set_title('CANAUX RGB - Prétraitées', fontweight='bold')
axes[1, 1].set_xlabel('Canal')
axes[1, 1].set_ylabel('Intensité moyenne (0-1)')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(channels)
axes[1, 1].grid(axis='y', alpha=0.3)
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('pixel_intensities_comparison.png', dpi=300, bbox_inches='tight')
print("✅ Graphique sauvegardé: pixel_intensities_comparison.png")
plt.show()

# ============================================================
# 6. COMPARAISON DÉTAILLÉE D'UNE SEULE IMAGE
# ============================================================

print("\n" + "="*70)
print("🔬 ANALYSE DÉTAILLÉE D'UNE IMAGE")
print("="*70)

# Prendre une image au hasard
class_name = classes[0]
raw_class_dir = os.path.join(RAW_TRAIN_DIR, class_name)
prep_class_dir = os.path.join(PREP_TRAIN_DIR, class_name)

images = [f for f in os.listdir(raw_class_dir) 
         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
img_name = images[0]

# Charger les images
raw_path = os.path.join(raw_class_dir, img_name)
prep_path = os.path.join(prep_class_dir, img_name)

raw_img = Image.open(raw_path).convert('RGB')
prep_img = Image.open(prep_path).convert('RGB')

raw_array = np.array(raw_img)
prep_array = np.array(prep_img)

print(f"\n📸 Image analysée: {img_name} ({class_name})")

# Créer la visualisation détaillée
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)

fig.suptitle(f'ANALYSE DÉTAILLÉE - {img_name}', fontsize=16, fontweight='bold')

# Image brute
ax1 = fig.add_subplot(gs[0, 0:2])
ax1.imshow(raw_img)
ax1.set_title(f'IMAGE BRUTE\n{raw_img.size[0]}x{raw_img.size[1]}px\nTaille: {os.path.getsize(raw_path)/(1024*1024):.3f}MB', 
              fontweight='bold', color='red')
ax1.axis('off')

# Image prétraitée
ax2 = fig.add_subplot(gs[0, 2:4])
ax2.imshow(prep_img)
ax2.set_title(f'IMAGE PRÉTRAITÉE\n{prep_img.size[0]}x{prep_img.size[1]}px\nTaille: {os.path.getsize(prep_path)/(1024*1024):.3f}MB', 
              fontweight='bold', color='green')
ax2.axis('off')

# Histogrammes RGB - Brute
ax3 = fig.add_subplot(gs[1, 0:2])
for i, color in enumerate(['red', 'green', 'blue']):
    hist, bins = np.histogram(raw_array[:,:,i], bins=256, range=(0, 255))
    ax3.plot(bins[:-1], hist, color=color, alpha=0.7, linewidth=2, label=color.upper())
ax3.set_title('HISTOGRAMME RGB - Brute', fontweight='bold')
ax3.set_xlabel('Valeur du pixel (0-255)')
ax3.set_ylabel('Fréquence')
ax3.legend()
ax3.grid(alpha=0.3)

# Histogrammes RGB - Prétraitée
ax4 = fig.add_subplot(gs[1, 2:4])
for i, color in enumerate(['red', 'green', 'blue']):
    hist, bins = np.histogram(prep_array[:,:,i], bins=256, range=(0, 255))
    ax4.plot(bins[:-1], hist, color=color, alpha=0.7, linewidth=2, label=color.upper())
ax4.set_title('HISTOGRAMME RGB - Prétraitée', fontweight='bold')
ax4.set_xlabel('Valeur du pixel (0-255)')
ax4.set_ylabel('Fréquence')
ax4.legend()
ax4.grid(alpha=0.3)

# Statistiques - Brute
ax5 = fig.add_subplot(gs[2, 0:2])
ax5.axis('off')
stats_text = f"""
STATISTIQUES - IMAGE BRUTE

Dimensions: {raw_img.size[0]} x {raw_img.size[1]} pixels
Nombre de pixels: {raw_img.size[0] * raw_img.size[1]:,}
Taille fichier: {os.path.getsize(raw_path)/(1024*1024):.3f} MB

Intensités (0-255):
  Rouge   - Min: {raw_array[:,:,0].min()}, Max: {raw_array[:,:,0].max()}, Moy: {raw_array[:,:,0].mean():.1f}
  Vert    - Min: {raw_array[:,:,1].min()}, Max: {raw_array[:,:,1].max()}, Moy: {raw_array[:,:,1].mean():.1f}
  Bleu    - Min: {raw_array[:,:,2].min()}, Max: {raw_array[:,:,2].max()}, Moy: {raw_array[:,:,2].mean():.1f}

Écart-type global: {raw_array.std():.2f}
"""
ax5.text(0.1, 0.5, stats_text, fontsize=10, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.3))

# Statistiques - Prétraitée
ax6 = fig.add_subplot(gs[2, 2:4])
ax6.axis('off')
stats_text = f"""
STATISTIQUES - IMAGE PRÉTRAITÉE

Dimensions: {prep_img.size[0]} x {prep_img.size[1]} pixels
Nombre de pixels: {prep_img.size[0] * prep_img.size[1]:,}
Taille fichier: {os.path.getsize(prep_path)/(1024*1024):.3f} MB

Intensités (0-255):
  Rouge   - Min: {prep_array[:,:,0].min()}, Max: {prep_array[:,:,0].max()}, Moy: {prep_array[:,:,0].mean():.1f}
  Vert    - Min: {prep_array[:,:,1].min()}, Max: {prep_array[:,:,1].max()}, Moy: {prep_array[:,:,1].mean():.1f}
  Bleu    - Min: {prep_array[:,:,2].min()}, Max: {prep_array[:,:,2].max()}, Moy: {prep_array[:,:,2].mean():.1f}

Écart-type global: {prep_array.std():.2f}
"""
ax6.text(0.1, 0.5, stats_text, fontsize=10, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))

plt.savefig('detailed_image_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Graphique sauvegardé: detailed_image_analysis.png")
plt.show()

# ============================================================
# 7. TABLEAU RÉCAPITULATIF DES AMÉLIORATIONS
# ============================================================

print("\n" + "="*70)
print("📋 TABLEAU RÉCAPITULATIF DES AMÉLIORATIONS")
print("="*70)

improvements = pd.DataFrame({
    'Métrique': [
        'Largeur moyenne (px)',
        'Hauteur moyenne (px)',
        'Tailles uniques',
        'Taille fichier moyenne (MB)',
        'Intensité moyenne (0-1)',
        'Écart-type intensité',
        'Standardisation'
    ],
    'Données Brutes': [
        f"{np.mean(raw_widths):.0f}",
        f"{np.mean(raw_heights):.0f}",
        f"{len(set(zip(raw_widths, raw_heights)))}",
        f"{np.mean(raw_sizes):.3f}",
        f"{np.mean(raw_means):.3f}",
        f"{np.std(raw_means):.3f}",
        "❌ Non"
    ],
    'Données Prétraitées': [
        f"{np.mean(prep_widths):.0f}",
        f"{np.mean(prep_heights):.0f}",
        f"{len(set(zip(prep_widths, prep_heights)))}",
        f"{np.mean(prep_sizes):.3f}",
        f"{np.mean(prep_means):.3f}",
        f"{np.std(prep_means):.3f}",
        "✅ Oui (224x224)"
    ],
    'Amélioration': [
        f"{((1 - abs(np.mean(prep_widths) - 224) / abs(np.mean(raw_widths) - 224)) * 100):.1f}%",
        f"{((1 - abs(np.mean(prep_heights) - 224) / abs(np.mean(raw_heights) - 224)) * 100):.1f}%",
        f"{((len(set(zip(raw_widths, raw_heights))) - 1) / len(set(zip(raw_widths, raw_heights))) * 100):.1f}%",
        f"{((np.mean(raw_sizes) - np.mean(prep_sizes)) / np.mean(raw_sizes) * 100):.1f}%",
        "Normalisé",
        f"{((np.std(raw_means) - np.std(prep_means)) / np.std(raw_means) * 100):.1f}%",
        "✅"
    ]
})

print("\n" + improvements.to_string(index=False))

# Sauvegarder en CSV
improvements.to_csv('preprocessing_improvements.csv', index=False)
print("\n✅ Tableau sauvegardé: preprocessing_improvements.csv")

# ============================================================
# 8. RAPPORT FINAL
# ============================================================

print("\n" + "="*70)
print("📊 RAPPORT FINAL - AMÉLIORATIONS APPORTÉES")
print("="*70)

print(f"\n✅ STANDARDISATION DES DIMENSIONS:")
print(f"   AVANT: Tailles variables ({len(set(zip(raw_widths, raw_heights)))} tailles différentes)")
print(f"   APRÈS: Taille unique (224x224 px)")
print(f"   → Permet l'utilisation de batch training")

print(f"\n✅ RÉDUCTION DE LA TAILLE DES FICHIERS:")
reduction_pct = ((np.mean(raw_sizes) - np.mean(prep_sizes)) / np.mean(raw_sizes)) * 100
print(f"   AVANT: {np.mean(raw_sizes):.3f} MB en moyenne")
print(f"   APRÈS: {np.mean(prep_sizes):.3f} MB en moyenne")
print(f"   → Réduction de {reduction_pct:.1f}%")
print(f"   → Chargement plus rapide")

print(f"\n✅ NORMALISATION DES INTENSITÉS:")
print(f"   AVANT: Intensité moyenne = {np.mean(raw_means):.3f} ± {np.std(raw_means):.3f}")
print(f"   APRÈS: Intensité moyenne = {np.mean(prep_means):.3f} ± {np.std(prep_means):.3f}")
print(f"   → Données plus homogènes")

print(f"\n✅ UNIFORMISATION DES CANAUX RGB:")
print(f"   AVANT: R={np.mean(raw_channels['R']):.3f}, G={np.mean(raw_channels['G']):.3f}, B={np.mean(raw_channels['B']):.3f}")
print(f"   APRÈS: R={np.mean(prep_channels['R']):.3f}, G={np.mean(prep_channels['G']):.3f}, B={np.mean(prep_channels['B']):.3f}")

# ============================================================
# 9. VISUALISATION FINALE - RÉSUMÉ GRAPHIQUE
# ============================================================

print("\n" + "="*70)
print("📈 CRÉATION DU GRAPHIQUE RÉSUMÉ")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('RÉSUMÉ DES AMÉLIORATIONS - PRÉTRAITEMENT', fontsize=18, fontweight='bold')

# 1. Comparaison des dimensions
ax = axes[0, 0]
categories = ['Largeur', 'Hauteur']
raw_dims = [np.mean(raw_widths), np.mean(raw_heights)]
prep_dims = [np.mean(prep_widths), np.mean(prep_heights)]
x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, raw_dims, width, label='Brutes', color='red', alpha=0.7)
bars2 = ax.bar(x + width/2, prep_dims, width, label='Prétraitées', color='green', alpha=0.7)
ax.axhline(y=224, color='blue', linestyle='--', linewidth=2, label='Taille cible (224px)')
ax.set_ylabel('Pixels')
ax.set_title('DIMENSIONS MOYENNES', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Ajouter les valeurs
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=10)

# 2. Comparaison taille fichier
ax = axes[0, 1]
sizes = ['Brutes', 'Prétraitées']
values = [np.mean(raw_sizes), np.mean(prep_sizes)]
colors = ['red', 'green']
bars = ax.bar(sizes, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Taille (MB)')
ax.set_title('TAILLE MOYENNE DES FICHIERS', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{bar.get_height():.3f} MB', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Annotation réduction
ax.annotate(f'Réduction\n{reduction_pct:.1f}%', xy=(1, values[1]), xytext=(1.3, values[0]/2),
            fontsize=12, fontweight='bold', color='blue',
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))

# 3. Distribution des intensités
ax = axes[1, 0]
bp1 = ax.boxplot([raw_means, prep_means], positions=[1, 2], widths=0.6,
                  patch_artist=True)
bp1['boxes'][0].set_facecolor('red')
bp1['boxes'][0].set_alpha(0.5)
bp1['boxes'][1].set_facecolor('green')
bp1['boxes'][1].set_alpha(0.5)
ax.set_xticklabels(['Brutes', 'Prétraitées'])
ax.set_ylabel('Intensité moyenne (0-1)')
ax.set_title('DISTRIBUTION DES INTENSITÉS', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# 4. Nombre de tailles uniques
ax = axes[1, 1]
unique_raw = len(set(zip(raw_widths, raw_heights)))
unique_prep = len(set(zip(prep_widths, prep_heights)))
categories = ['Brutes', 'Prétraitées']
values = [unique_raw, unique_prep]
colors = ['red', 'green']
bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Nombre de tailles différentes')
ax.set_title('UNIFORMITÉ DES DIMENSIONS', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=14, fontweight='bold')

# Annotation
ax.annotate('✅ Standardisé!', xy=(1, unique_prep), xytext=(1.2, unique_raw/2),
            fontsize=12, fontweight='bold', color='green',
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

plt.tight_layout()
plt.savefig('preprocessing_summary.png', dpi=300, bbox_inches='tight')
print("✅ Graphique sauvegardé: preprocessing_summary.png")
plt.show()

# ============================================================
# 10. FICHIERS GÉNÉRÉS
# ============================================================

print("\n" + "="*70)
print("💾 FICHIERS GÉNÉRÉS")
print("="*70)

print(f"""
📁 Graphiques:
   - dimensions_comparison.png        → Comparaison des dimensions
   - visual_comparison_by_class.png   → Échantillons visuels par classe
   - pixel_intensities_comparison.png → Analyse des intensités
   - detailed_image_analysis.png      → Analyse détaillée d'une image
   - preprocessing_summary.png        → Résumé graphique

📊 Données:
   - preprocessing_improvements.csv   → Tableau des améliorations
""")

print("\n" + "="*70)
print("🎉 VISUALISATION COMPARÉE TERMINÉE!")
print("="*70)

print(f"""
📝 RÉSUMÉ DES AMÉLIORATIONS:

1. ✅ DIMENSIONS STANDARDISÉES
   • Toutes les images sont maintenant 224x224 px
   • Permet le batch training efficace
   
2. ✅ TAILLE DES FICHIERS RÉDUITE
   • Réduction de {reduction_pct:.1f}%
   • Chargement plus rapide
   • Moins de RAM utilisée

3. ✅ INTENSITÉS NORMALISÉES
   • Distribution plus homogène
   • Meilleure convergence du modèle

4. ✅ UNIFORMITÉ GARANTIE
   • {unique_raw} tailles → {unique_prep} taille unique
   • Compatible avec tous les réseaux de neurones

💡 PROCHAINE ÉTAPE:
   Lancer le notebook d'entraînement avec les données prétraitées!
"")